# 🚀 Space Rock Shooter — 이미지 적용 버전

기존 게임(`kdt-final-3-game-subject`)의 코드를 그대로 복사한 뒤, `images` 폴더의 그림을 입혀서 만든 노트북 버전입니다.
**이 노트북은 게임 저장소 폴더(`kdt-final-3-game-subject`, = `images`·`maps` 폴더가 있는 곳)에서 열고, 위에서부터 차례로 모든 셀을 실행하세요.** 마지막 셀을 실행하면 게임 창이 열립니다.

## 이미지 → 게임 요소

| 역할 | images 폴더 파일 | 게임에서 |
|---|---|---|
| **bg** | `KakaoTalk_20261014_222840803.jpg` | 게임 배경 화면 |
| **guard** | `KakaoTalk_20261014_222850176_01.png` (남색 바위) | 블록 2~5개를 가로/세로로 이어 붙인(가끔 ㄱ자) 엄폐물을 맵 곳곳에 랜덤 배치. **총알과 우주선이 통과 못 함** |
| **rock_01** | `KakaoTalk_20261014_222850176_02.png` (연보라) | 가장 작은 운석 · **5점** · 총알 5발 · 가장 많이 나옴 (70%) |
| **rock_02** | `KakaoTalk_20261014_222850176_03.png` (살구색) | 중간 운석 · **15점** · 총알 **15발** · 두 번째로 많이 나옴 (30%) |
| **rock_03** | `KakaoTalk_20261014_222850176_04.png` (주황) | 가장 큰 운석 · **30점** · 총알 **45발** · 랜덤으로는 안 나오고, **운석을 3개 부술 때마다 1개** 플레이어 근처에 등장 |
| **spaceship** | `KakaoTalk_20261014_222850176.png` | 플레이어. 이미지 **위쪽 포구에서 총알**이 나감 |

> 파일 이름을 `bg.jpg`, `guard.png`, `rock_01.png` … 처럼 바꿔 넣으면 그 파일을 먼저 사용합니다.

## 조작

| 키 | 동작 |
|---|---|
| **W / A / S / D** | 위 / 왼쪽 / 아래 / 오른쪽 이동 |
| 마우스 커서 | 조준 (우주선 이미지가 커서 쪽으로 돌아감) |
| **Space** 또는 마우스 왼쪽 버튼 | 발사 (누르고 있으면 연사) |
| R | 재장전 · L / 1·2·3: 레벨업 카드 · ESC/P: 메뉴 · N: 엄폐물 새로 배치 · F1: 디버그 |

## 셀 구성
0. import + 이미지 경로 → 1. config → 2. terrain → 3. progression/menus → 4. game_core → 5. renderer → 6. main → 7. **실행**

각 셀 맨 위 주석에 원본에서 **[변경] / [추가]** 된 부분을 적어 두었습니다. (필요 패키지: `pygame`, `numpy` — 없으면 아래 셀의 `#`를 지우고 실행)


In [1]:
# 처음 한 번만 필요하면 아래 줄의 # 를 지우고 실행하세요
# %pip install pygame numpy

In [2]:
# ==========================================================================
# 0) 공통 import + 이미지 경로 설정
#    - 이 노트북은 게임 저장소 폴더('kdt-final-3-game-subject') 안에
#      images/, maps/ 폴더와 함께 들어 있습니다. 이 폴더에서 열어 실행하세요.
#    - images 폴더 파일 이름을 bg.png / guard.png / rock_01.png ... 처럼 바꿔도
#      자동으로 그 파일을 먼저 찾습니다.
# ==========================================================================
import os
import sys
import math
import random
import heapq
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pygame
import pygame.gfxdraw

# 노트북이 있는 폴더 = 게임 저장소 폴더 (images/, maps/ 가 바로 아래에 있음)
BASE_DIR = Path.cwd()
if not (BASE_DIR / "images").is_dir() and (BASE_DIR / "kdt-final-3-game-subject" / "images").is_dir():
    # 혹시 상위 폴더('동그리빵야')에서 실행한 경우에도 찾아갈 수 있게
    BASE_DIR = BASE_DIR / "kdt-final-3-game-subject"
IMAGES_DIR = BASE_DIR / "images"
GAME_DIR = BASE_DIR                                 # 게임 폴더 (maps/ 맵 txt 파일 위치)

# 역할 이름 → images 폴더의 실제 파일 이름 (카카오톡에서 받은 이름 그대로)
IMAGE_FILES = {
    "bg":        "KakaoTalk_20261014_222840803.jpg",     # 배경
    "spaceship": "KakaoTalk_20261014_222850176.png",     # 플레이어 우주선
    "guard":     "KakaoTalk_20261014_222850176_01.png",  # 엄폐물 블록 (남색 바위)
    "rock_01":   "KakaoTalk_20261014_222850176_02.png",  # 작은 운석 (연보라)
    "rock_02":   "KakaoTalk_20261014_222850176_03.png",  # 중간 운석 (살구색)
    "rock_03":   "KakaoTalk_20261014_222850176_04.png",  # 대형 운석 (주황)
}


def image_path(key: str) -> Path:
    """images/<key>.png 같은 이름이 있으면 그걸 쓰고, 없으면 IMAGE_FILES의 파일을 쓴다."""
    for ext in (".png", ".jpg", ".jpeg", ".webp"):
        p = IMAGES_DIR / f"{key}{ext}"
        if p.exists():
            return p
    return IMAGES_DIR / IMAGE_FILES[key]


_missing = [k for k in IMAGE_FILES if not image_path(k).exists()]
print("이미지 폴더:", IMAGES_DIR)
for _k in IMAGE_FILES:
    print(f"  {_k:<10} → {image_path(_k).name}")
if _missing:
    print("⚠ 찾을 수 없는 이미지:", _missing, "- 노트북을 'kdt-final-3-game-subject' 폴더에서 열었는지 확인하세요.")

pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html
이미지 폴더: c:\AI코딩\동그리빵야\kdt-final-3-game-subject\images
  bg         → KakaoTalk_20261014_222840803.jpg
  spaceship  → KakaoTalk_20261014_222850176.png
  guard      → KakaoTalk_20261014_222850176_01.png
  rock_01    → KakaoTalk_20261014_222850176_02.png
  rock_02    → KakaoTalk_20261014_222850176_03.png
  rock_03    → KakaoTalk_20261014_222850176_04.png


In [3]:
# ==========================================================================
# 1) config.py (복사 + 수정)
#    [변경] 도형 3종 → 운석 rock_01 / rock_02 / rock_03
#           hp = "부수는 데 필요한 총알 수" (총알 1발 = 1칸)
#    [변경] 엄폐물(guard) 랜덤 배치 설정, 우주선 이미지 관련 설정 추가
# ==========================================================================


def resource_path(relative: str) -> str:
    """원본 게임 폴더 기준 경로 (노트북에는 __file__이 없으므로 GAME_DIR 사용)."""
    return os.path.join(str(GAME_DIR), relative)


# --------------------------------------------------------------------------
# 운석 종류: 체력(=필요 총알 수), 점수, 반지름, 생성 가중치
#   rock_01: 가장 작음, 5점, 가장 많이 나옴
#   rock_02: 중간, 15점, 총알 15발, 두 번째로 많이 나옴
#   rock_03: 가장 큼, 30점, 총알 45발, 랜덤 생성 없음 → 운석 3개를 부술 때마다 1개 등장
# --------------------------------------------------------------------------
SHAPE_TYPES = {
    "rock_01": {"hp": 5.0,  "score": 5,  "radius": 17, "weight": 0.70},
    "rock_02": {"hp": 15.0, "score": 15, "radius": 25, "weight": 0.30},
    "rock_03": {"hp": 45.0, "score": 30, "radius": 46, "weight": 0.0},
}
SHAPE_KINDS = list(SHAPE_TYPES.keys())
BIG_ROCK = "rock_03"
MAX_SHAPE_SCORE = max(t["score"] for t in SHAPE_TYPES.values())
MAX_SHAPE_RADIUS = max(t["radius"] for t in SHAPE_TYPES.values())


@dataclass
class EnemyAIConfig:
    """규칙 기반 적 AI 난이도 파라미터."""
    reaction_frames: int = 20        # 목표/이동 방향을 다시 정하는 주기 (프레임)
    aim_noise_deg: float = 12.0      # 조준 오차 표준편차 (도)
    turn_speed_scale: float = 0.7    # 플레이어 포신 회전 속도 대비 배율
    move_speed_scale: float = 0.8    # 플레이어 가속도 대비 배율
    fire_cooldown: int = 36          # 발사 간격 (프레임)
    fire_angle_tol_deg: float = 15.0 # 조준 오차가 이 값 이내일 때만 발사
    player_priority: float = 1.6     # 플레이어까지 거리를 이 값으로 나눠서 비교 (클수록 플레이어 우선)
    sight_range: float = 450.0       # 이 거리 밖의 플레이어는 무시
    keep_distance: float = 230.0     # 플레이어와 유지하려는 거리
    respawn_frames: int = 300        # 사망 후 재등장까지 프레임 (5초)
    memory_frames: int = 120         # 플레이어를 놓친 뒤 마지막 위치를 기억하는 시간 (2초)
    path_recompute_frames: int = 20  # A* 경로 재계산 주기
    waypoint_reach: float = 14.0     # 경로의 다음 칸 중심에 이만큼 가까워지면 다음 칸으로


@dataclass
class TerrainConfig:
    # 화면 표현: 벽 = 엄폐물(guard 이미지 블록)
    enabled: bool = True             # False면 벽 없는 빈 경기장
    tile_size: int = 48              # 기본 맵 템플릿 960x720 → 20x15 타일
    map_name: str = "drift"          # maps/<map_name>.txt (guard_map=False일 때만 사용)
    map_pool: tuple = ()
    random_map: bool = False
    # [추가] 엄폐물 랜덤 배치: guard 블록을 2~5칸씩 이어 붙인 구조물을 곳곳에 놓는다
    guard_map: bool = True
    guard_density: float = 0.05      # 전체 칸 중 엄폐물 비율 (5%)
    guard_min_len: int = 2           # 구조물 한 줄의 최소 블록 수
    guard_max_len: int = 5           # 구조물 한 줄의 최대 블록 수
    guard_bend_chance: float = 0.35  # ㄱ자로 꺾일 확률
    guard_clear_radius: int = 3      # 시작 위치 주변 비워둘 칸 수
    random_wall_density: float = 0.06
    random_bush_density: float = 0.0     # 가스 구름은 사용하지 않음 (이미지 없음)
    random_mud_density: float = 0.03
    random_breakable_density: float = 0.03
    mud_enabled: bool = False
    mud_accel_scale: float = 0.5
    breakable_enabled: bool = False
    breakable_hp: float = 40.0


@dataclass
class GameConfig:
    # 화면 / 월드 / 시간
    width: int = 960
    height: int = 720
    world_scale: int = 3             # 2880x2160 월드, 60x45 타일
    camera_zoom: float = 0.6
    progression_enabled: bool = True
    level_xp_base: int = 40          # [변경] 운석 점수가 작아져서 60 → 40
    level_xp_growth: int = 25        # [변경] 40 → 25
    fps: int = 60

    # 탱크(우주선) 공통
    tank_radius: float = 22.0
    barrel_length: float = 30.0      # [변경] 우주선 이미지 맨 위(포구)에서 총알이 나가도록
    barrel_width: float = 16.0
    accel: float = 0.35
    friction: float = 0.90
    turn_speed: float = 0.07
    body_damage: float = 0.6         # 접촉 시 프레임당 데미지 (운석은 몸통 충돌로 깎이지 않음)

    # 플레이어
    player_hp: float = 100.0
    player_fire_cooldown: int = 8    # [변경] 20 → 8 (45발짜리 운석을 부수려면 연사가 빨라야 함)
    limited_ammo: bool = True
    player_ammo: int = 120
    reload_seconds: float = 2.5
    regen_delay: int = 240
    regen_rate: float = 0.05
    fixed_up_aim: bool = False       # [추가] True면 마우스 조준 없이 항상 화면 위쪽으로만 발사

    # 적
    num_enemies: int = 9
    enemy_hp: float = 60.0
    enemy_ai: EnemyAIConfig = field(default_factory=EnemyAIConfig)
    enemy_kill_score: int = 200

    # 지형
    terrain: TerrainConfig = field(default_factory=TerrainConfig)

    # 탄환
    bullet_speed: float = 9.0
    bullet_radius: float = 6.0
    bullet_life: int = 70
    bullet_damage: float = 7.0       # 적 우주선에 주는 피해 (운석은 총알 1발 = 1칸)

    # 운석
    num_shapes: int = 72             # rock_01 + rock_02 유지 개수
    shape_drift: float = 0.25
    spawn_margin: float = 40.0
    spawn_min_dist_from_tanks: float = 120.0
    big_rock_every: int = 3          # [추가] 운석을 이만큼 부술 때마다 rock_03 1개 등장
    big_rock_max_alive: int = 3      # [추가] rock_03 동시 최대 개수
    big_spawn_min_dist: float = 320.0  # [추가] rock_03은 플레이어 근처(이 거리 사이)에 등장
    big_spawn_max_dist: float = 650.0

    # [추가] 이미지 크기 (충돌 반지름 대비 배율)
    ship_draw_scale: float = 2.5     # 우주선 이미지 가로 = 반지름 × 2.5
    rock_draw_scale: float = 2.35    # 운석 이미지 긴 변 = 반지름 × 2.35
    guard_overlap: int = 10          # 엄폐물 블록을 타일보다 크게 그려 서로 이어 보이게

    @property
    def world_width(self) -> int:
        return self.width * self.world_scale

    @property
    def world_height(self) -> int:
        return self.height * self.world_scale

    @property
    def max_speed(self) -> float:
        return self.accel / (1.0 - self.friction)

In [4]:
# ==========================================================================
# 2) terrain.py (복사 + 수정)
#    타일 지형, 벽 충돌, 시야선, A* 경로 탐색은 원본 그대로
#    [추가] generate_guard_map: guard 블록을 이어 붙인 엄폐물을 랜덤 배치
# ==========================================================================

EMPTY, WALL, BUSH, MUD, BREAKABLE = 0, 1, 2, 3, 4
CHAR_TO_TILE = {".": EMPTY, "#": WALL, "*": BUSH, "~": MUD, "+": BREAKABLE, "P": EMPTY, "E": EMPTY}
TILE_TO_CHAR = {EMPTY: ".", WALL: "#", BUSH: "*", MUD: "~", BREAKABLE: "+"}

# 8방향 이웃 (dc, dr, 비용)
_NEIGHBORS = [(1, 0, 1.0), (-1, 0, 1.0), (0, 1, 1.0), (0, -1, 1.0),
              (1, 1, math.sqrt(2)), (1, -1, math.sqrt(2)), (-1, 1, math.sqrt(2)), (-1, -1, math.sqrt(2))]


class MapError(ValueError):
    pass


def grid_channels(tcfg: TerrainConfig):
    """관찰 격자에 들어가는 지형 채널 이름 (설정에 따라 고정)."""
    ch = ["wall", "bush"]
    if tcfg.mud_enabled:
        ch.append("mud")
    if tcfg.breakable_enabled:
        ch.append("breakable")
    return ch


def list_maps():
    d = resource_path("maps")
    if not os.path.isdir(d):
        return []
    return sorted(f[:-4] for f in os.listdir(d) if f.endswith(".txt"))


class TileMap:
    def __init__(self, tiles, tile_size, player_start, enemy_starts, name, tcfg: TerrainConfig):
        self.rows = len(tiles)
        self.cols = len(tiles[0])
        self.tile = tile_size
        self.name = name
        self.player_start = player_start          # (col, row)
        self.enemy_starts = list(enemy_starts)    # [(col, row), ...]
        self.breakable_hp_max = tcfg.breakable_hp
        # 설정에서 끈 기능은 해당 타일을 일반 타일로 취급
        conv = {MUD: MUD if tcfg.mud_enabled else EMPTY,
                BREAKABLE: BREAKABLE if tcfg.breakable_enabled else WALL}
        self.tiles = [[conv.get(t, t) for t in row] for row in tiles]
        self.hp = {(c, r): tcfg.breakable_hp for r in range(self.rows) for c in range(self.cols)
                   if self.tiles[r][c] == BREAKABLE}
        self.version = 0          # 지형이 바뀔 때마다 증가 (렌더러/관찰 캐시 무효화용)
        self._refresh()

    def _refresh(self):
        self.solid = [[t in (WALL, BREAKABLE) for t in row] for row in self.tiles]
        self.has_solid = any(any(row) for row in self.solid)
        # 자신/주변 8칸(맵 바깥 포함)에 막힌 칸이 있는지. 물체 반지름 < 타일 크기라
        # 이 값이 False인 칸에 중심이 있으면 벽 충돌 검사를 건너뛸 수 있다.
        self.near_solid = [[any(self.is_solid_tile(c + dc, r + dr) for dc in (-1, 0, 1) for dr in (-1, 0, 1))
                            for c in range(self.cols)] for r in range(self.rows)]
        self.free_tiles = [(c, r) for r in range(self.rows) for c in range(self.cols) if not self.solid[r][c]]
        # 사방이 막히지 않은 칸 (큰 도형 스폰용)
        self.open_tiles = [(c, r) for c, r in self.free_tiles
                           if all(not self.is_solid_tile(c + dc, r + dr)
                                  for dc, dr in ((1, 0), (-1, 0), (0, 1), (0, -1)))]
        self.version += 1

    # ---------------- 조회 ----------------
    def is_solid_tile(self, c, r):
        return c < 0 or r < 0 or c >= self.cols or r >= self.rows or self.solid[r][c]

    def tile_of(self, x, y):
        return int(x // self.tile), int(y // self.tile)

    def tile_at(self, x, y):
        c, r = int(x // self.tile), int(y // self.tile)
        if c < 0 or r < 0 or c >= self.cols or r >= self.rows:
            return WALL
        return self.tiles[r][c]

    def is_solid_at(self, x, y):
        return self.is_solid_tile(int(x // self.tile), int(y // self.tile))

    def center(self, c, r):
        return (c + 0.5) * self.tile, (r + 0.5) * self.tile

    def as_array(self):
        return np.array(self.tiles, dtype=np.int8)

    # ---------------- 부서지는 벽 ----------------
    def damage_tile(self, c, r, dmg) -> bool:
        """부서지는 벽에 데미지. 파괴되면 True."""
        key = (c, r)
        if key not in self.hp:
            return False
        self.hp[key] -= dmg
        if self.hp[key] <= 0:
            del self.hp[key]
            self.tiles[r][c] = EMPTY
            self._refresh()
            return True
        self.version += 1
        return False

    # ---------------- 충돌 ----------------
    def resolve_circle(self, obj, bounce=False) -> bool:
        """원(obj.x, obj.y, obj.radius)을 벽 밖으로 밀어낸다.
        벽을 향하는 속도 성분만 제거(bounce면 반사)하므로 벽을 따라 미끄러진다."""
        if not self.has_solid:
            return False
        T = self.tile
        r = obj.radius
        tc0, tr0 = int(obj.x // T), int(obj.y // T)
        if 0 <= tc0 < self.cols and 0 <= tr0 < self.rows and not self.near_solid[tr0][tc0] and r < T:
            return False   # 주변에 벽이 없음 (대부분의 경우 여기서 끝)
        hit = False
        for _ in range(2):  # 모서리에서 두 벽에 동시에 닿는 경우 대비
            moved = False
            c0, c1 = int((obj.x - r) // T), int((obj.x + r) // T)
            r0, r1 = int((obj.y - r) // T), int((obj.y + r) // T)
            for tr in range(r0, r1 + 1):
                for tc in range(c0, c1 + 1):
                    if not self.is_solid_tile(tc, tr):
                        continue
                    left, top = tc * T, tr * T
                    px = min(max(obj.x, left), left + T)
                    py = min(max(obj.y, top), top + T)
                    dx, dy = obj.x - px, obj.y - py
                    d2 = dx * dx + dy * dy
                    if d2 >= r * r:
                        continue
                    if d2 > 1e-9:
                        d = math.sqrt(d2)
                        nx, ny = dx / d, dy / d
                        push = r - d
                    else:  # 중심이 벽 안: 가장 얕은 축으로 밀기
                        pens = [(obj.x - left, -1.0, 0.0), (left + T - obj.x, 1.0, 0.0),
                                (obj.y - top, 0.0, -1.0), (top + T - obj.y, 0.0, 1.0)]
                        pen, nx, ny = min(pens)
                        push = pen + r
                    obj.x += nx * push
                    obj.y += ny * push
                    vn = obj.vx * nx + obj.vy * ny
                    if vn < 0:
                        k = 2.0 if bounce else 1.0
                        obj.vx -= k * vn * nx
                        obj.vy -= k * vn * ny
                    moved = hit = True
            if not moved:
                break
        return hit

    # ---------------- 시야 ----------------
    def raycast(self, x0, y0, dx, dy, max_dist) -> float:
        """(x0,y0)에서 단위벡터 (dx,dy) 방향으로 첫 벽까지 거리 (없으면 max_dist). 격자 DDA."""
        T = self.tile
        c, r = int(x0 // T), int(y0 // T)
        if self.is_solid_tile(c, r):
            return 0.0
        step_c = 1 if dx > 0 else -1
        step_r = 1 if dy > 0 else -1
        inf = float("inf")
        t_dc = abs(T / dx) if dx != 0 else inf
        t_dr = abs(T / dy) if dy != 0 else inf
        t_c = (((c + 1) * T - x0) / dx if dx > 0 else (x0 - c * T) / -dx) if dx != 0 else inf
        t_r = (((r + 1) * T - y0) / dy if dy > 0 else (y0 - r * T) / -dy) if dy != 0 else inf
        while True:
            if t_c < t_r:
                t, c = t_c, c + step_c
                t_c += t_dc
            else:
                t, r = t_r, r + step_r
                t_r += t_dr
            if t >= max_dist:
                return max_dist
            if self.is_solid_tile(c, r):
                return t

    def line_of_sight(self, x0, y0, x1, y1) -> bool:
        if not self.has_solid:
            return True
        dx, dy = x1 - x0, y1 - y0
        d = math.hypot(dx, dy)
        if d < 1e-6:
            return True
        return self.raycast(x0, y0, dx / d, dy / d, d) >= d

    # ---------------- 경로 ----------------
    def astar(self, start, goal, max_nodes=2000):
        """8방향 A* (벽 모서리 대각선 통과 금지). 타일 좌표 리스트 반환 (start 제외), 실패 시 []."""
        if start == goal or self.is_solid_tile(*goal):
            return []
        gx, gy = goal

        def h(c, r):
            ax, ay = abs(c - gx), abs(r - gy)
            return max(ax, ay) + (math.sqrt(2) - 1) * min(ax, ay)

        open_heap = [(h(*start), 0, 0.0, start)]
        came = {start: None}
        g = {start: 0.0}
        counter = 0
        while open_heap and counter < max_nodes:
            _, _, cost, cur = heapq.heappop(open_heap)
            if cur == goal:
                path = []
                while cur != start:
                    path.append(cur)
                    cur = came[cur]
                return path[::-1]
            if cost > g[cur]:
                continue
            c, r = cur
            for dc, dr, w in _NEIGHBORS:
                nc, nr = c + dc, r + dr
                if self.is_solid_tile(nc, nr):
                    continue
                if dc and dr and (self.is_solid_tile(c + dc, r) or self.is_solid_tile(c, r + dr)):
                    continue
                ng = cost + w
                if ng < g.get((nc, nr), float("inf")):
                    g[(nc, nr)] = ng
                    came[(nc, nr)] = cur
                    counter += 1
                    heapq.heappush(open_heap, (ng + h(nc, nr), counter, ng, (nc, nr)))
        return []

    def reachable_from(self, start):
        """4방향 BFS로 start에서 갈 수 있는 빈칸 집합."""
        seen = {start}
        q = deque([start])
        while q:
            c, r = q.popleft()
            for dc, dr in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                n = (c + dc, r + dr)
                if n not in seen and not self.is_solid_tile(*n):
                    seen.add(n)
                    q.append(n)
        return seen

    def is_connected(self):
        return len(self.reachable_from(self.player_start)) == len(self.free_tiles)


# --------------------------------------------------------------------------
# 맵 로드 / 생성
# --------------------------------------------------------------------------
def parse_map(text, name, tcfg: TerrainConfig, cols, rows) -> TileMap:
    lines = [ln.rstrip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln and not ln.startswith(";")]  # ';'로 시작하면 주석
    if len(lines) != rows:
        raise MapError(f"[{name}] 행 수가 {len(lines)}개입니다. {rows}개여야 합니다 (화면 {cols}x{rows} 타일).")
    tiles, player, enemies = [], None, []
    for r, ln in enumerate(lines):
        if len(ln) != cols:
            raise MapError(f"[{name}] {r + 1}번째 줄 길이가 {len(ln)}입니다. {cols}이어야 합니다: '{ln}'")
        row = []
        for c, ch in enumerate(ln):
            if ch not in CHAR_TO_TILE:
                raise MapError(f"[{name}] {r + 1}번째 줄 {c + 1}번째 문자 '{ch}'는 알 수 없는 지형입니다.")
            if ch == "P":
                if player is not None:
                    raise MapError(f"[{name}] P(플레이어 시작)가 여러 개입니다.")
                player = (c, r)
            elif ch == "E":
                enemies.append((c, r))
            row.append(CHAR_TO_TILE[ch])
        tiles.append(row)
    if player is None:
        raise MapError(f"[{name}] P(플레이어 시작)가 없습니다.")
    tm = TileMap(tiles, tcfg.tile_size, player, enemies, name, tcfg)
    if not tm.is_connected():
        unreachable = set(tm.free_tiles) - tm.reachable_from(player)
        raise MapError(f"[{name}] 플레이어 시작점에서 갈 수 없는 빈칸이 있습니다: {sorted(unreachable)[:5]}...")
    return tm


_MAP_TEXT_CACHE = {}


def load_map(name, tcfg: TerrainConfig, cols, rows) -> TileMap:
    if name not in _MAP_TEXT_CACHE:
        path = resource_path(os.path.join("maps", f"{name}.txt"))
        if not os.path.exists(path):
            raise MapError(f"맵 파일이 없습니다: {path} (사용 가능: {list_maps()})")
        with open(path, encoding="utf-8") as f:
            _MAP_TEXT_CACHE[name] = f.read()
    return parse_map(_MAP_TEXT_CACHE[name], name, tcfg, cols, rows)


def empty_map(tcfg: TerrainConfig, cols, rows) -> TileMap:
    tiles = [[EMPTY] * cols for _ in range(rows)]
    return TileMap(tiles, tcfg.tile_size, (cols // 2, rows // 2), [], "empty", tcfg)


def expand_map(tm: TileMap, scale: int, tcfg: TerrainConfig) -> TileMap:
    """원본 타일 크기를 유지하고 맵을 반복 배치한다. 시작점은 중앙 구역에 둔다."""
    if scale == 1:
        return tm
    tiles = [[tm.tiles[r % tm.rows][c % tm.cols] for c in range(tm.cols * scale)]
             for r in range(tm.rows * scale)]
    player = (tm.player_start[0] + (scale // 2) * tm.cols,
              tm.player_start[1] + (scale // 2) * tm.rows)
    enemies = [(c + sx * tm.cols, r + sy * tm.rows)
               for sy in range(scale) for sx in range(scale) for c, r in tm.enemy_starts]
    expanded = TileMap(tiles, tm.tile, player, enemies, tm.name, tcfg)
    if not expanded.is_connected():
        raise MapError(f"[{tm.name}] 확장된 맵의 구역들이 연결되지 않습니다.")
    return expanded


def generate_random_map(rng, tcfg: TerrainConfig, cols, rows, max_attempts=50) -> TileMap:
    """랜덤 맵 생성. BFS로 모든 빈칸이 연결될 때까지 다시 생성 (끝내 실패하면 고립된 칸을 벽으로 메움)."""
    p = (cols // 2, rows // 2)
    tiles = None
    for _ in range(max_attempts):
        tiles = [[EMPTY] * cols for _ in range(rows)]

        def stamp(kind, density, wmax, hmax, only_empty):
            target = int(density * cols * rows)
            count, guard = 0, 0
            while count < target and guard < 500:
                guard += 1
                w, h = int(rng.integers(1, wmax + 1)), int(rng.integers(1, hmax + 1))
                c0, r0 = int(rng.integers(0, cols - w + 1)), int(rng.integers(0, rows - h + 1))
                for r in range(r0, r0 + h):
                    for c in range(c0, c0 + w):
                        if (not only_empty or tiles[r][c] == EMPTY) and tiles[r][c] != kind:
                            tiles[r][c] = kind
                            count += 1

        stamp(WALL, tcfg.random_wall_density, 3, 3, False)
        stamp(BUSH, tcfg.random_bush_density, 3, 2, True)
        if tcfg.mud_enabled:
            stamp(MUD, tcfg.random_mud_density, 3, 3, True)
        if tcfg.breakable_enabled:
            stamp(BREAKABLE, tcfg.random_breakable_density, 2, 1, True)
        for r in range(p[1] - 1, p[1] + 2):          # 시작점 주변 비우기
            for c in range(p[0] - 1, p[0] + 2):
                tiles[r][c] = EMPTY
        tm = TileMap(tiles, tcfg.tile_size, p, [], "random", tcfg)
        if tm.is_connected():
            break
    else:
        reach = tm.reachable_from(p)
        for c, r in tm.free_tiles:
            if (c, r) not in reach:
                tiles[r][c] = WALL
        tm = TileMap(tiles, tcfg.tile_size, p, [], "random", tcfg)

    # 적 시작점: 플레이어에서 먼 빈칸 중 랜덤 4개
    far = [t for t in tm.free_tiles if abs(t[0] - p[0]) + abs(t[1] - p[1]) >= 10 and tm.tiles[t[1]][t[0]] == EMPTY]
    if far:
        idx = rng.choice(len(far), size=min(4, len(far)), replace=False)
        tm.enemy_starts = [far[int(i)] for i in idx]
    return tm


def map_to_text(tm: TileMap) -> str:
    rows = []
    for r in range(tm.rows):
        line = "".join(TILE_TO_CHAR[t] for t in tm.tiles[r])
        rows.append(line)
    pc, pr = tm.player_start
    rows[pr] = rows[pr][:pc] + "P" + rows[pr][pc + 1:]
    for ec, er in tm.enemy_starts:
        rows[er] = rows[er][:ec] + "E" + rows[er][ec + 1:]
    return "\n".join(rows)


# --------------------------------------------------------------------------
# [추가] 엄폐물(guard) 랜덤 배치
#   guard 블록 2~5개를 가로 또는 세로로 이어 붙이고, 가끔 ㄱ자로 꺾어서
#   맵 곳곳에 구조물을 만든다. 구조물끼리는 1칸 이상 띄워서 길이 막히지 않게 한다.
# --------------------------------------------------------------------------
def generate_guard_map(rng, tcfg: TerrainConfig, cols, rows, max_attempts=50) -> TileMap:
    p = (cols // 2, rows // 2)
    clear = tcfg.guard_clear_radius
    tiles = None
    tm = None
    for _ in range(max_attempts):
        tiles = [[EMPTY] * cols for _ in range(rows)]
        target = int(tcfg.guard_density * cols * rows)
        count, tries = 0, 0
        while count < target and tries < 3000:
            tries += 1
            length = int(rng.integers(tcfg.guard_min_len, tcfg.guard_max_len + 1))
            horizontal = bool(rng.random() < 0.5)
            dc, dr = (1, 0) if horizontal else (0, 1)
            c0 = int(rng.integers(1, cols - 1))
            r0 = int(rng.integers(1, rows - 1))
            cells = [(c0 + dc * i, r0 + dr * i) for i in range(length)]
            if rng.random() < tcfg.guard_bend_chance:          # ㄱ자로 꺾기
                ec, er = cells[-1]
                bend = int(rng.integers(1, 3))
                sign = 1 if rng.random() < 0.5 else -1
                cells += [(ec + dr * sign * i, er + dc * sign * i) for i in range(1, bend + 1)]
            # 맵 가장자리 1칸은 비워둔다 (벽에 붙어 막힌 공간이 생기지 않게)
            if any(not (1 <= c < cols - 1 and 1 <= r < rows - 1) for c, r in cells):
                continue
            # 시작 위치 주변은 비워둔다
            if any(abs(c - p[0]) <= clear and abs(r - p[1]) <= clear for c, r in cells):
                continue
            # 다른 구조물과 붙지 않게 (주변 8칸에 이미 엄폐물이 있으면 건너뜀)
            mine = set(cells)
            if any(tiles[r + rr][c + cc] == WALL
                   for c, r in cells for cc in (-1, 0, 1) for rr in (-1, 0, 1)
                   if (c + cc, r + rr) not in mine):
                continue
            for c, r in cells:
                tiles[r][c] = WALL
            count += len(cells)
        tm = TileMap(tiles, tcfg.tile_size, p, [], "guard", tcfg)
        if tm.is_connected():
            break
    else:
        reach = tm.reachable_from(p)
        for c, r in tm.free_tiles:
            if (c, r) not in reach:
                tiles[r][c] = WALL
        tm = TileMap(tiles, tcfg.tile_size, p, [], "guard", tcfg)

    # 적 시작점: 플레이어에서 먼 빈칸 중 랜덤 12개
    far = [t for t in tm.free_tiles if abs(t[0] - p[0]) + abs(t[1] - p[1]) >= 14]
    if far:
        idx = rng.choice(len(far), size=min(12, len(far)), replace=False)
        tm.enemy_starts = [far[int(i)] for i in idx]
    return tm

In [5]:
# ==========================================================================
# 3) progression.py + menus.py (원본 그대로 복사)
#    레벨업 보상 카드, 메인/설정 메뉴 버튼
# ==========================================================================

@dataclass(frozen=True)
class RewardCard:
    key: str
    title: str
    category: str
    description: tuple[str, ...]


REWARD_CARDS = (
    RewardCard("damage", "화력 강화", "강화", ("탄환 피해량 +25%",)),
    RewardCard("rapid", "속사 장치", "강화", ("발사 대기 시간 -15%", "최소 4프레임까지 강화")),
    RewardCard("speed", "추진기 강화", "능력치", ("이동 가속도 +15%",)),
    RewardCard("hull", "선체 확장", "능력치", ("최대 체력 +25", "현재 체력도 +25")),
    RewardCard("heal", "긴급 수리", "회복", ("최대 체력의 60% 회복",)),
    RewardCard("regen", "자가 수리", "회복", ("초당 자동 회복량 +1.5", "피격 후 회복 대기 유지")),
    RewardCard("range", "고속 탄환", "강화", ("탄환 속도 +20%", "탄환 수명 +15%")),
    RewardCard("scout", "정찰선 전직", "전직", ("이동 가속도 +35%", "탄환 속도 +20%")),
    RewardCard("gunner", "포격선 전직", "전직", ("3방향 동시 발사", "각 탄환 피해량 -20%")),
    RewardCard("guardian", "수호선 전직", "전직", ("최대 체력 +60, 즉시 완전 회복", "초당 자동 회복량 +1.5")),
)

JOB_NAMES = {"rookie": "기본 우주선", "scout": "정찰선", "gunner": "포격선", "guardian": "수호선"}


@dataclass(frozen=True)
class MenuButton:
    key: str
    label: str
    rect: pygame.Rect
    enabled: bool = True


def menu_buttons(mode, cfg, show_debug=False, show_controls=True, auto_pause=True):
    x = cfg.width // 2 - 190
    if mode == "main":
        y = cfg.height // 2 - 107
        return [MenuButton("start", "무한 스테이지", pygame.Rect(x, y, 380, 58)),
                MenuButton("story", "스토리 모드 · 준비 중", pygame.Rect(x, y + 78, 380, 58), False),
                MenuButton("quit", "게임 나가기", pygame.Rect(x, y + 156, 380, 58))]
    rows = [("zoom", f"줌 배율   {cfg.camera_zoom:.0%}"),
            ("controls", f"조작 안내   {'켜짐' if show_controls else '꺼짐'}"),
            ("debug", f"디버그 표시   {'켜짐' if show_debug else '꺼짐'}"),
            ("auto_pause", f"창 전환 시 자동 일시정지   {'켜짐' if auto_pause else '꺼짐'}")]
    y = cfg.height // 2 - 160
    buttons = [MenuButton(key, label, pygame.Rect(x, y + i * 46, 380, 38))
               for i, (key, label) in enumerate(rows)]
    actions = [("resume", "계속하기"), ("restart", "새 게임"),
               ("main", "메인 메뉴로"), ("quit", "게임 나가기")]
    buttons.extend(MenuButton(key, label, pygame.Rect(x, y + 206 + i * 48, 380, 40))
                   for i, (key, label) in enumerate(actions))
    return buttons

In [6]:
# ==========================================================================
# 4) game_core.py (복사 + 수정)
#    [변경] 운석 체력 = 필요한 총알 수 → 총알 1발이 맞으면 1칸 감소
#    [변경] 몸통으로 부딪혀도 운석은 깎이지 않음 (우주선만 피해)
#    [추가] 운석을 3개 부술 때마다 rock_03(대형)을 플레이어 근처에 1개 등장
#    [추가] 엄폐물 랜덤 맵, 우주선은 이미지 위쪽(포구)이 위를 보며 시작
# ==========================================================================

# --------------------------------------------------------------------------
# 행동 정의: 이동 9 × 회전 3 = 27
# --------------------------------------------------------------------------
# move: 0=정지, 1=↑, 2=↗, 3=→, 4=↘, 5=↓, 6=↙, 7=←, 8=↖   (화면 좌표: y가 아래로 증가)
_S = 1.0 / math.sqrt(2.0)
MOVE_VECTORS = [
    (0.0, 0.0), (0.0, -1.0), (_S, -_S), (1.0, 0.0), (_S, _S),
    (0.0, 1.0), (-_S, _S), (-1.0, 0.0), (-_S, -_S),
]
N_MOVES = 9
N_ROTS = 3          # 0=유지, 1=왼쪽(반시계), 2=오른쪽(시계)
N_ACTIONS = N_MOVES * N_ROTS
ROT_DIR = (0.0, -1.0, 1.0)


def encode_action(move: int, rot: int) -> int:
    return move * N_ROTS + rot


def decode_action(action: int):
    action = int(action)
    if not 0 <= action < N_ACTIONS:
        raise ValueError(f"action must be in [0, {N_ACTIONS}), got {action}")
    return divmod(action, N_ROTS)


def move_from_vector(dx: float, dy: float) -> int:
    """임의 방향 벡터를 가장 가까운 8방향 이동 번호로 양자화 (적 AI가 같은 행동 집합을 쓰도록)."""
    if dx * dx + dy * dy < 1e-6:
        return 0
    ang = math.atan2(dy, dx)                       # 0=→, +π/2=↓
    sector = int(round(ang / (math.pi / 4))) % 8   # 0=→,1=↘,2=↓,3=↙,4=←,5=↖,6=↑,7=↗
    return (3, 4, 5, 6, 7, 8, 1, 2)[sector]


def wrap_angle(a: float) -> float:
    return (a + math.pi) % (2 * math.pi) - math.pi


# --------------------------------------------------------------------------
# 엔티티
# --------------------------------------------------------------------------
@dataclass
class Tank:
    id: int
    team: str            # "player" | "enemy"
    x: float
    y: float
    hp: float
    max_hp: float
    radius: float
    fire_cooldown_max: int
    vx: float = 0.0
    vy: float = 0.0
    angle: float = 0.0   # 포신 각도 (rad, 0=→, 시계방향 +)
    cooldown: int = 0
    alive: bool = True
    respawn_timer: int = 0
    last_hit_frame: int = -10**9
    damage_scale: float = 1.0
    accel_scale: float = 1.0
    bullet_speed_scale: float = 1.0
    bullet_life_scale: float = 1.0
    regen_bonus: float = 0.0
    projectiles: int = 1
    job: str = "rookie"
    ammo: int = 0
    max_ammo: int = 0
    # 적 AI 상태
    ai_timer: int = 0
    ai_target: tuple = None      # ("player", id) | ("shape", id) | None
    ai_aim_offset: float = 0.0
    ai_move: int = 0
    ai_can_see: bool = False          # 이번 프레임에 플레이어가 보이는지
    ai_last_seen: tuple = None        # 마지막으로 본 플레이어 위치 (x, y)
    ai_last_seen_frame: int = -10**9
    ai_path: list = field(default_factory=list)   # 남은 경로 (타일 좌표)
    ai_path_timer: int = 0
    ai_goal: tuple = None             # 현재 이동 목표 위치 (디버그 표시용)
    ai_strafe: float = 1.0            # 옆으로 돌 방향


@dataclass
class Shape:
    id: int
    kind: str
    x: float
    y: float
    vx: float
    vy: float
    hp: float
    max_hp: float
    radius: float
    score: int
    angle: float = 0.0
    spin: float = 0.0


@dataclass
class Bullet:
    x: float
    y: float
    vx: float
    vy: float
    life: int
    owner_id: int
    team: str
    damage: float
    radius: float


def empty_events():
    return {
        "shape_damage_weighted": 0.0,  # Σ 데미지 × (점수/최대체력)
        "shape_destroyed_score": 0.0,  # Σ 파괴한 도형 점수
        "shapes_destroyed": 0,
        "enemy_damage": 0.0,
        "enemies_killed": 0,
        "damage_taken": 0.0,
        "died": False,
    }


# --------------------------------------------------------------------------
# 게임
# --------------------------------------------------------------------------
class GameCore:
    def __init__(self, config: GameConfig = None, seed: int = None):
        self.cfg = config or GameConfig()
        if not isinstance(self.cfg.world_scale, int) or self.cfg.world_scale < 1:
            raise ValueError("world_scale은 1 이상의 정수여야 합니다.")
        if self.cfg.level_xp_base <= 0 or self.cfg.level_xp_growth < 0:
            raise ValueError("레벨 경험치는 양수, 증가량은 0 이상이어야 합니다.")
        if not isinstance(self.cfg.player_ammo, int) or self.cfg.player_ammo < 0:
            raise ValueError("player_ammo는 0 이상의 정수여야 합니다.")
        if self.cfg.reload_seconds <= 0 or self.cfg.fps <= 0:
            raise ValueError("reload_seconds와 fps는 양수여야 합니다.")
        self.reset(seed)

    # ---------------- 초기화 ----------------
    def reset(self, seed: int = None):
        self.rng = np.random.default_rng(seed)
        # 카드 열기/선택이 맵 생성과 전투 난수열을 바꾸지 않도록 분리한다.
        self.reward_rng = np.random.default_rng(seed)
        self.frame = 0
        self.score = 0
        self.kills = 0
        self.game_over = False
        self._next_id = 0
        self.events = empty_events()
        self.level = 1
        self.xp = 0
        self.pending_rewards = 0
        self.reward_choices = ()
        self.last_reward = ""
        self.reward_message_until = 0
        self.reload_frames_left = 0
        # [추가] 대형 운석(rock_03) 등장 규칙: 운석 3개 파괴 → 1개 등장
        self.rocks_destroyed = 0       # 이번 판에 플레이어가 부순 운석 수
        self.big_progress = 0          # 다음 rock_03까지 모은 파괴 수 (0~2)
        self.pending_big_rocks = 0     # 등장 대기 중인 rock_03 수
        self.big_message_until = 0
        self.bullets = []
        self.shapes = []
        self.enemies = []   # 스폰 위치 계산이 이전 판의 적을 참조하지 않도록 먼저 비움

        c = self.cfg
        self.terrain = self._make_terrain()
        px, py = self.terrain.center(*self.terrain.player_start)
        self.player = Tank(
            id=self._new_id(), team="player", x=px, y=py,
            hp=c.player_hp, max_hp=c.player_hp, radius=c.tank_radius,
            fire_cooldown_max=c.player_fire_cooldown,
            ammo=c.player_ammo, max_ammo=c.player_ammo,
            angle=-math.pi / 2,   # [변경] 이미지 위쪽(포구)이 화면 위를 보며 시작
        )
        for _ in range(c.num_enemies):
            e = Tank(id=self._new_id(), team="enemy", x=0.0, y=0.0,
                     hp=c.enemy_hp, max_hp=c.enemy_hp, radius=c.tank_radius,
                     fire_cooldown_max=c.enemy_ai.fire_cooldown)
            self._respawn_enemy(e)
            self.enemies.append(e)
        for _ in range(c.num_shapes):
            self.shapes.append(self._make_shape())

    def _new_id(self) -> int:
        self._next_id += 1
        return self._next_id

    def _make_terrain(self) -> TileMap:
        c = self.cfg
        tc = c.terrain
        cols, rows = c.width // tc.tile_size, c.height // tc.tile_size
        if cols * tc.tile_size != c.width or rows * tc.tile_size != c.height:
            raise ValueError(f"화면 크기 {c.width}x{c.height}가 타일 크기 {tc.tile_size}로 나누어떨어지지 않습니다.")
        if not tc.enabled:
            return empty_map(tc, cols * c.world_scale, rows * c.world_scale)
        if tc.guard_map:   # [추가] 엄폐물 랜덤 배치
            return generate_guard_map(self.rng, tc, cols * c.world_scale, rows * c.world_scale)
        if tc.random_map:
            return generate_random_map(self.rng, tc, cols * c.world_scale, rows * c.world_scale)
        name = tc.map_name
        if tc.map_pool:
            name = tc.map_pool[int(self.rng.integers(len(tc.map_pool)))]
        return expand_map(load_map(name, tc, cols, rows), c.world_scale, tc)

    def is_hidden(self, t) -> bool:
        """수풀 안에 있는 탱크는 상대에게 보이지 않는다."""
        return self.terrain.tile_at(t.x, t.y) == BUSH

    def _random_free_pos(self, radius: float, min_dist: float, tiles=None):
        """벽이 아닌 칸 중에서 탱크들과 min_dist 이상 떨어진 위치."""
        tm = self.terrain
        T = tm.tile
        if tiles is None:
            tiles = tm.open_tiles if radius > T / 2 - 2 and tm.open_tiles else tm.free_tiles
        tanks = [self.player] + [e for e in getattr(self, "enemies", []) if e.alive]
        jitter = max(0.0, T / 2 - radius)
        x = y = 0.0
        for _ in range(20):
            tc, tr = tiles[int(self.rng.integers(len(tiles)))]
            cx, cy = tm.center(tc, tr)
            x = cx + float(self.rng.uniform(-jitter, jitter))
            y = cy + float(self.rng.uniform(-jitter, jitter))
            if all((t.x - x) ** 2 + (t.y - y) ** 2 >= min_dist ** 2 for t in tanks):
                break
        return x, y

    def _make_shape(self, kind=None, near_player=False) -> Shape:
        """[변경] kind를 주면 그 운석을, 안 주면 가중치(rock_01 70% / rock_02 30%)로 고른다."""
        c = self.cfg
        if kind is None:
            weights = np.array([SHAPE_TYPES[k]["weight"] for k in SHAPE_KINDS])
            kind = SHAPE_KINDS[int(self.rng.choice(len(SHAPE_KINDS), p=weights / weights.sum()))]
        t = SHAPE_TYPES[kind]
        if near_player:
            x, y = self._pos_near_player(t["radius"])
        else:
            x, y = self._random_free_pos(t["radius"], c.spawn_min_dist_from_tanks)
        ang = float(self.rng.uniform(-math.pi, math.pi))
        spd = float(self.rng.uniform(0, c.shape_drift))
        return Shape(id=self._new_id(), kind=kind, x=x, y=y,
                     vx=math.cos(ang) * spd, vy=math.sin(ang) * spd,
                     hp=t["hp"], max_hp=t["hp"], radius=t["radius"], score=t["score"],
                     angle=float(self.rng.uniform(0, 2 * math.pi)),
                     spin=float(self.rng.uniform(-0.01, 0.01)))

    def _pos_near_player(self, radius):
        """[추가] 플레이어에게서 big_spawn_min_dist ~ big_spawn_max_dist 떨어진 빈칸 (화면 안쪽에 보이도록)."""
        c = self.cfg
        p = self.player
        tm = self.terrain
        lo, hi = c.big_spawn_min_dist ** 2, c.big_spawn_max_dist ** 2
        tiles = [t for t in (tm.open_tiles or tm.free_tiles)
                 if lo <= (tm.center(*t)[0] - p.x) ** 2 + (tm.center(*t)[1] - p.y) ** 2 <= hi]
        if not tiles:
            return self._random_free_pos(radius, c.spawn_min_dist_from_tanks)
        return self._random_free_pos(radius, radius + p.radius + 20, tiles=tiles)

    def _respawn_enemy(self, e: Tank):
        tm = self.terrain
        p = self.player
        # 맵의 E 위치 중 플레이어와 멀고 다른 적과 겹치지 않는 곳 우선
        others = [o for o in getattr(self, "enemies", []) if o.alive and o is not e]
        starts = []
        for s in tm.enemy_starts:
            sx, sy = tm.center(*s)
            if (sx - p.x) ** 2 + (sy - p.y) ** 2 >= 300.0 ** 2 and \
                    all((sx - o.x) ** 2 + (sy - o.y) ** 2 >= (2 * e.radius) ** 2 for o in others):
                starts.append((sx, sy))
        if starts:
            e.x, e.y = starts[int(self.rng.integers(len(starts)))]
        else:
            e.x, e.y = self._random_free_pos(e.radius, 300.0)
        e.vx = e.vy = 0.0
        e.hp = e.max_hp
        e.alive = True
        e.cooldown = e.fire_cooldown_max
        e.angle = float(self.rng.uniform(-math.pi, math.pi))
        e.ai_timer = 0
        e.ai_target = None
        e.ai_can_see = False
        e.ai_last_seen = None
        e.ai_path = []
        e.ai_path_timer = 0
        e.ai_goal = None

    # ---------------- 외부 API ----------------
    @property
    def xp_to_next_level(self):
        return self.cfg.level_xp_base + (self.level - 1) * self.cfg.level_xp_growth

    def _add_score(self, amount):
        self.score += amount
        if not self.cfg.progression_enabled:
            return
        self.xp += amount
        while self.xp >= self.xp_to_next_level:
            self.xp -= self.xp_to_next_level
            self.level += 1
            self.pending_rewards += 1

    def open_rewards(self):
        if self.game_over or not self.player.alive or self.pending_rewards == 0:
            return ()
        if not self.reward_choices:
            eligible = [card for card in REWARD_CARDS
                        if card.category != "전직" or (self.level >= 3 and self.player.job == "rookie")]
            indices = self.reward_rng.choice(len(eligible), size=3, replace=False)
            self.reward_choices = tuple(eligible[int(i)] for i in indices)
        return self.reward_choices

    def choose_reward(self, index):
        if self.game_over or not self.player.alive or not self.reward_choices:
            return False
        if not isinstance(index, int) or not 0 <= index < len(self.reward_choices):
            return False
        card = self.reward_choices[index]
        p = self.player
        if card.key == "damage":
            p.damage_scale *= 1.25
        elif card.key == "rapid":
            p.fire_cooldown_max = max(4, int(p.fire_cooldown_max * 0.85))
            p.cooldown = min(p.cooldown, p.fire_cooldown_max)
        elif card.key == "speed":
            p.accel_scale *= 1.15
        elif card.key == "hull":
            p.max_hp += 25
            p.hp = min(p.max_hp, p.hp + 25)
        elif card.key == "heal":
            p.hp = min(p.max_hp, p.hp + p.max_hp * 0.6)
        elif card.key == "regen":
            p.regen_bonus += 1.5 / self.cfg.fps
        elif card.key == "range":
            p.bullet_speed_scale *= 1.2
            p.bullet_life_scale *= 1.15
        elif card.key == "scout":
            p.job = "scout"
            p.accel_scale *= 1.35
            p.bullet_speed_scale *= 1.2
        elif card.key == "gunner":
            p.job = "gunner"
            p.projectiles = 3
            p.damage_scale *= 0.8
        elif card.key == "guardian":
            p.job = "guardian"
            p.max_hp += 60
            p.hp = p.max_hp
            p.regen_bonus += 1.5 / self.cfg.fps
        self.pending_rewards -= 1
        self.reward_choices = ()
        self.last_reward = card.title
        self.reward_message_until = self.frame + self.cfg.fps * 3
        return True

    @property
    def reload_frames_total(self):
        return math.ceil(self.cfg.reload_seconds * self.cfg.fps)

    def request_reload(self):
        if (self.game_over or not self.player.alive or not self.cfg.limited_ammo
                or self.reload_frames_left or self.player.ammo >= self.player.max_ammo):
            return False
        self.reload_frames_left = self.reload_frames_total
        return True

    def pop_events(self) -> dict:
        ev = self.events
        self.events = empty_events()
        return ev

    def step(self, action: int, *, want_fire: bool = True, aim_position=None):
        """1프레임 진행. 사람 입력은 발사 여부와 월드 좌표 조준점을 별도로 전달한다."""
        if self.game_over:
            return
        move, rot = decode_action(action)
        self._apply_tank_control(self.player, move, rot, self.player.accel_scale, 1.0)
        if self.cfg.fixed_up_aim:          # [추가] 항상 위쪽으로만 발사하는 모드
            self.player.angle = -math.pi / 2
        elif aim_position is not None:
            dx, dy = aim_position[0] - self.player.x, aim_position[1] - self.player.y
            if dx * dx + dy * dy > 1e-6:
                self.player.angle = math.atan2(dy, dx)

        for e in self.enemies:
            if e.alive:
                self._enemy_ai(e)
            else:
                e.respawn_timer -= 1
                if e.respawn_timer <= 0:
                    self._respawn_enemy(e)

        self._fire(self.player, want_fire)
        self._move_shapes()
        self._move_bullets()
        self._collide_bullets()
        self._collide_bodies()
        self._regen_player()

        # [변경] 운석 개수 유지: 대기 중인 rock_03 먼저 등장 → rock_01/02는 num_shapes개 유지
        c = self.cfg
        n_big = sum(s.kind == BIG_ROCK for s in self.shapes)
        while self.pending_big_rocks > 0 and n_big < c.big_rock_max_alive:
            self.shapes.append(self._make_shape(BIG_ROCK, near_player=True))
            self.pending_big_rocks -= 1
            n_big += 1
            self.big_message_until = self.frame + c.fps * 2
        while len(self.shapes) - n_big < c.num_shapes:
            self.shapes.append(self._make_shape())

        self.frame += 1
        if self.reload_frames_left:
            self.reload_frames_left -= 1
            if self.reload_frames_left == 0 and self.player.hp > 0:
                self.player.ammo = self.player.max_ammo
        if self.player.hp <= 0:
            self.player.hp = 0.0
            self.player.alive = False
            self.game_over = True
            self.events["died"] = True

    # ---------------- 탱크 이동/발사 ----------------
    def _apply_tank_control(self, t: Tank, move: int, rot: int, accel_scale: float, turn_scale: float):
        c = self.cfg
        if self.terrain.tile_at(t.x, t.y) == MUD:
            accel_scale *= c.terrain.mud_accel_scale
        mx, my = MOVE_VECTORS[move]
        t.vx = t.vx * c.friction + mx * c.accel * accel_scale
        t.vy = t.vy * c.friction + my * c.accel * accel_scale
        t.x += t.vx
        t.y += t.vy
        self.terrain.resolve_circle(t)
        self._clamp_to_arena(t)
        t.angle = wrap_angle(t.angle + ROT_DIR[rot] * c.turn_speed * turn_scale)

    def _clamp_to_arena(self, obj):
        c = self.cfg
        r = obj.radius
        if obj.x < r:
            obj.x, obj.vx = r, max(0.0, obj.vx)
        elif obj.x > c.world_width - r:
            obj.x, obj.vx = c.world_width - r, min(0.0, obj.vx)
        if obj.y < r:
            obj.y, obj.vy = r, max(0.0, obj.vy)
        elif obj.y > c.world_height - r:
            obj.y, obj.vy = c.world_height - r, min(0.0, obj.vy)

    def _fire(self, t: Tank, want_fire: bool):
        if t.cooldown > 0:
            t.cooldown -= 1
            return
        if not want_fire:
            return
        c = self.cfg
        if t.team == "player" and self.reload_frames_left:
            return
        offsets = (-0.20, 0.0, 0.20) if t.projectiles == 3 else (0.0,)
        if t.team == "player" and c.limited_ammo:
            # 마지막 탄약이 3발 미만이면 남은 수만큼 발사한다.
            if t.ammo <= 0:
                return
            offsets = offsets[:t.ammo]
            t.ammo -= len(offsets)
        for offset in offsets:
            ca, sa = math.cos(t.angle + offset), math.sin(t.angle + offset)
            self.bullets.append(Bullet(
                x=t.x + ca * c.barrel_length, y=t.y + sa * c.barrel_length,
                vx=ca * c.bullet_speed * t.bullet_speed_scale,
                vy=sa * c.bullet_speed * t.bullet_speed_scale,
                life=int(c.bullet_life * t.bullet_life_scale), owner_id=t.id, team=t.team,
                damage=c.bullet_damage * t.damage_scale, radius=c.bullet_radius,
            ))
        t.cooldown = t.fire_cooldown_max

    # ---------------- 적 AI ----------------
    def _enemy_ai(self, e: Tank):
        c = self.cfg
        ai = c.enemy_ai
        p = self.player
        tm = self.terrain

        # 1) 감지: 사거리 안 + 수풀 밖 + 시야선이 트였을 때만 플레이어를 본다
        e.ai_can_see = (p.alive
                        and (p.x - e.x) ** 2 + (p.y - e.y) ** 2 <= ai.sight_range ** 2
                        and not self.is_hidden(p)
                        and tm.line_of_sight(e.x, e.y, p.x, p.y))
        chasing_player = e.ai_target is not None and e.ai_target[0] in ("player", "point")
        if e.ai_can_see and chasing_player:   # 플레이어를 쫓는 중일 때만 위치를 기억
            e.ai_last_seen = (p.x, p.y)
            e.ai_last_seen_frame = self.frame
        elif e.ai_last_seen is not None and self.frame - e.ai_last_seen_frame > ai.memory_frames:
            e.ai_last_seen = None

        # 2) 목표 선택 (reaction_frames마다): 기억한 위치 추적 중이면 계속, 아니면 플레이어/도형 가중 거리 비교
        e.ai_timer -= 1
        if e.ai_timer <= 0:
            e.ai_timer = ai.reaction_frames
            if e.ai_last_seen is not None and not e.ai_can_see:
                target = ("point", e.ai_last_seen)
            else:
                # 보이는 플레이어와 가장 가까운 도형을 가중 거리로 비교 (player_priority)
                target, best_d = None, float("inf")
                if e.ai_can_see:
                    target, best_d = ("player", p.id), math.hypot(p.x - e.x, p.y - e.y) / ai.player_priority
                for s in self.shapes:
                    d = math.hypot(s.x - e.x, s.y - e.y)
                    if d < best_d:
                        target, best_d = ("shape", s.id), d
            if target != e.ai_target:
                e.ai_path_timer = 0          # 목표가 바뀌면 경로 즉시 재계산
            e.ai_target = target
            e.ai_aim_offset = float(self.rng.normal(0.0, math.radians(ai.aim_noise_deg)))
            e.ai_strafe = 1.0 if self.rng.random() < 0.5 else -1.0
            if target is None:
                e.ai_move = int(self.rng.integers(0, N_MOVES))

        tgt = self._target_pos(e)
        if tgt is None and e.ai_target is not None and e.ai_target[0] != "player":
            e.ai_target = None

        # 3) 이동
        direct = False
        move = e.ai_move if e.ai_target is None else 0
        e.ai_goal = tgt
        if tgt is not None:
            tx, ty = tgt
            kind = e.ai_target[0]
            direct = kind != "point" and tm.line_of_sight(e.x, e.y, tx, ty)
            if direct:
                # 직접 보이면: 적정 거리 유지 + 옆으로 돌기
                e.ai_path = []
                dx, dy = tx - e.x, ty - e.y
                d = math.hypot(dx, dy) + 1e-6
                keep = ai.keep_distance if kind == "player" else 140.0
                if d > keep + 40:
                    vx, vy = dx / d, dy / d
                elif d < keep - 40:
                    vx, vy = -dx / d, -dy / d
                else:
                    vx, vy = -dy / d * e.ai_strafe, dx / d * e.ai_strafe
            else:
                # 안 보이면: A* 경로를 따라감
                e.ai_path_timer -= 1
                if e.ai_path_timer <= 0:
                    e.ai_path_timer = ai.path_recompute_frames
                    e.ai_path = tm.astar(tm.tile_of(e.x, e.y), tm.tile_of(tx, ty))
                while e.ai_path:
                    wx, wy = tm.center(*e.ai_path[0])
                    if (wx - e.x) ** 2 + (wy - e.y) ** 2 < ai.waypoint_reach ** 2:
                        e.ai_path.pop(0)
                    else:
                        break
                if e.ai_path:
                    wx, wy = tm.center(*e.ai_path[0])
                    vx, vy = wx - e.x, wy - e.y
                else:
                    vx, vy = tx - e.x, ty - e.y
                    if kind == "point" and vx * vx + vy * vy < (tm.tile * 0.5) ** 2:
                        e.ai_last_seen = None    # 마지막 위치에 왔는데 없음 → 포기
                        e.ai_timer = 0
            move = move_from_vector(vx, vy)

        # 4) 조준/발사: 목표가 시야에 있을 때만 쏜다
        rot = 0
        aligned = False
        if tgt is not None:
            desired = math.atan2(tgt[1] - e.y, tgt[0] - e.x) + e.ai_aim_offset
            diff = wrap_angle(desired - e.angle)
            step = c.turn_speed * ai.turn_speed_scale
            if diff > step:
                rot = 2
            elif diff < -step:
                rot = 1
            aligned = abs(diff) <= math.radians(ai.fire_angle_tol_deg)
        kind = e.ai_target[0] if e.ai_target else None
        can_fire = (kind == "player" and e.ai_can_see) or (kind == "shape" and direct)

        self._apply_tank_control(e, move, rot, ai.move_speed_scale, ai.turn_speed_scale)
        self._fire(e, aligned and can_fire)

    def _target_pos(self, e: Tank):
        """적 AI 목표의 현재 위치. 플레이어가 안 보이면 마지막으로 본 위치를 쓴다."""
        t = e.ai_target
        if t is None:
            return None
        kind, val = t
        if kind == "player":
            if not self.player.alive:
                return None
            return (self.player.x, self.player.y) if e.ai_can_see else e.ai_last_seen
        if kind == "point":
            return val
        for s in self.shapes:
            if s.id == val:
                return (s.x, s.y)
        return None

    # ---------------- 도형/탄환 ----------------
    def _move_shapes(self):
        c = self.cfg
        for s in self.shapes:
            s.x += s.vx
            s.y += s.vy
            s.angle += s.spin
            if s.x < s.radius or s.x > c.world_width - s.radius:
                s.vx = -s.vx
            if s.y < s.radius or s.y > c.world_height - s.radius:
                s.vy = -s.vy
            self.terrain.resolve_circle(s, bounce=True)
            self._clamp_to_arena(s)
            # 밀려난 속도는 서서히 표류 속도로 감쇠
            spd = math.hypot(s.vx, s.vy)
            if spd > c.shape_drift:
                s.vx *= 0.95
                s.vy *= 0.95

    def _move_bullets(self):
        c = self.cfg
        tm = self.terrain
        alive = []
        for b in self.bullets:
            b.x += b.vx
            b.y += b.vy
            b.life -= 1
            if not (b.life > 0 and b.radius <= b.x <= c.world_width - b.radius and b.radius <= b.y <= c.world_height - b.radius):
                continue
            if tm.has_solid and tm.is_solid_at(b.x, b.y):   # 벽에 닿으면 소멸
                tc, tr = tm.tile_of(b.x, b.y)
                if tm.tiles[tr][tc] == BREAKABLE:
                    tm.damage_tile(tc, tr, b.damage)
                continue
            alive.append(b)
        self.bullets = alive

    def _collide_bullets(self):
        remaining = []
        for b in self.bullets:
            if not self._bullet_hit(b):
                remaining.append(b)
        self.bullets = remaining
        self.shapes = [s for s in self.shapes if s.hp > 0]

    def _bullet_hit(self, b: Bullet) -> bool:
        """탄환이 무언가에 맞았으면 데미지 처리 후 True."""
        bx, by, br = b.x, b.y, b.radius
        by_player = b.team == "player"
        # 탱크
        targets = self.enemies if by_player else (self.player,)
        for t in targets:
            r = br + t.radius
            if t.alive and (t.x - bx) ** 2 + (t.y - by) ** 2 < r * r:
                self._damage_tank(t, b.damage, by_player=by_player)
                return True
        # 도형 (호출 수가 많아 거리 계산을 인라인)
        for s in self.shapes:
            r = br + s.radius
            if (s.x - bx) ** 2 + (s.y - by) ** 2 < r * r and s.hp > 0:
                self._damage_shape(s, 1.0, by_player=by_player)   # [변경] 총알 1발 = 운석 체력 1칸
                return True
        return False

    def _damage_tank(self, t: Tank, dmg: float, by_player: bool):
        dealt = min(dmg, t.hp)
        if dealt <= 0:
            return
        t.hp -= dealt
        t.last_hit_frame = self.frame
        if t.team == "player":
            self.events["damage_taken"] += dealt
            return
        if by_player:
            self.events["enemy_damage"] += dealt
        if t.hp <= 0:
            t.alive = False
            t.respawn_timer = self.cfg.enemy_ai.respawn_frames
            if by_player:
                self.events["enemies_killed"] += 1
                self.kills += 1
                self._add_score(self.cfg.enemy_kill_score)

    def _damage_shape(self, s: Shape, dmg: float, by_player: bool):
        dealt = min(dmg, s.hp)
        if dealt <= 0:
            return
        s.hp -= dealt
        if by_player:
            self.events["shape_damage_weighted"] += dealt * s.score / s.max_hp
            if s.hp <= 0:
                self.events["shape_destroyed_score"] += s.score
                self.events["shapes_destroyed"] += 1
                self._add_score(s.score)
                # [추가] 운석 3개를 부술 때마다 rock_03 1개 등장 예약
                self.rocks_destroyed += 1
                self.big_progress += 1
                if self.big_progress >= self.cfg.big_rock_every:
                    self.big_progress -= self.cfg.big_rock_every
                    self.pending_big_rocks += 1

    # ---------------- 몸체 충돌 ----------------
    def _collide_bodies(self):
        bd = self.cfg.body_damage
        tanks = [self.player] + [e for e in self.enemies if e.alive]
        for t in tanks:
            for s in self.shapes:
                r = t.radius + s.radius
                if (s.x - t.x) ** 2 + (s.y - t.y) ** 2 < r * r and s.hp > 0:
                    _push_apart(t, s, 0.3)   # [변경] 운석은 몸통 충돌로 깎이지 않음
                    self._damage_tank(t, bd, by_player=False)
        p = self.player
        for e in self.enemies:
            if e.alive and _overlap(p, e):
                _push_apart(p, e, 0.5)
                self._damage_tank(e, bd, by_player=True)
                self._damage_tank(p, bd, by_player=False)
        for i, a in enumerate(self.enemies):
            for b in self.enemies[i + 1:]:
                if a.alive and b.alive and _overlap(a, b):
                    _push_apart(a, b, 0.5)
        self.shapes = [s for s in self.shapes if s.hp > 0]
        # 서로 밀어내다 벽에 들어간 경우 다시 밖으로
        for o in tanks + self.shapes:
            self.terrain.resolve_circle(o)
            self._clamp_to_arena(o)

    def _regen_player(self):
        p = self.player
        c = self.cfg
        if p.hp > 0 and self.frame - p.last_hit_frame >= c.regen_delay:
            p.hp = min(p.max_hp, p.hp + c.regen_rate + p.regen_bonus)


def _overlap(a, b) -> bool:
    r = a.radius + b.radius
    dx, dy = a.x - b.x, a.y - b.y
    return dx * dx + dy * dy < r * r


def _push_apart(a, b, a_share: float):
    """겹친 만큼 밀어낸다. a_share는 a가 움직이는 비율."""
    dx, dy = a.x - b.x, a.y - b.y
    d = math.hypot(dx, dy)
    if d < 1e-6:
        dx, dy, d = 1.0, 0.0, 1.0
    overlap = a.radius + b.radius - d
    nx, ny = dx / d, dy / d
    a.x += nx * overlap * a_share
    a.y += ny * overlap * a_share
    b.x -= nx * overlap * (1 - a_share)
    b.y -= ny * overlap * (1 - a_share)
    # 밀려나는 방향으로 약한 반동
    a.vx += nx * 0.5 * a_share
    a.vy += ny * 0.5 * a_share
    b.vx -= nx * 0.5 * (1 - a_share)
    b.vy -= ny * 0.5 * (1 - a_share)

In [7]:
# ==========================================================================
# 5) renderer.py (복사 + 수정)
#    [변경] 배경 = bg 이미지, 엄폐물 = guard 이미지 블록, 운석 = rock_01~03 이미지
#    [변경] 플레이어 = spaceship 이미지 (이미지 위쪽 포구에서 총알이 나감)
#    적 우주선, 총알, 폭발 연출, HUD, 메뉴는 원본 그대로
# ==========================================================================

COLORS = {
    "space_top": (6, 8, 22),
    "space_bottom": (16, 10, 34),
    "player": (70, 200, 255),
    "player_dark": (18, 84, 140),
    "enemy": (255, 92, 112),
    "enemy_dark": (130, 28, 56),
    "player_shot": (90, 220, 255),
    "enemy_shot": (255, 96, 160),
    "square": (255, 220, 110),      # 별 조각
    "triangle": (232, 124, 86),     # 암석 행성
    "pentagon": (128, 118, 255),    # 고리 행성
    "rock_01": (196, 186, 236),     # [추가] 운석 파괴 연출 색
    "rock_02": (244, 178, 140),
    "rock_03": (252, 170, 80),
    "nebula": [(150, 70, 220), (80, 96, 240), (210, 78, 190)],
    "cloud": (34, 62, 94),
    "cloud_edge": (92, 150, 196),
    "gravity": (255, 150, 60),
    "rock": (122, 110, 102),
    "rock_edge": (70, 62, 58),
    "text": (180, 192, 226),
    "label": (130, 150, 205),
    "white": (255, 255, 255),
    "panel": (14, 20, 42),
    "panel_edge": (78, 98, 160),
    "hp_bg": (36, 40, 66),
}
SPAWN_FRAMES = 14   # 등장 애니메이션 길이
DEATH_FRAMES = 22   # 파괴/폭발 애니메이션 길이
FLASH_FRAMES = 5    # 피격 깜빡임 길이

# 우주선 선체 모양 (반지름 단위, 기수가 +x)
PLAYER_HULL = [(1.25, 0), (0.35, 0.32), (-0.15, 0.95), (-0.6, 0.95), (-0.45, 0.35),
               (-0.62, 0), (-0.45, -0.35), (-0.6, -0.95), (-0.15, -0.95), (0.35, -0.32)]
ENEMY_HULL = [(1.15, 0), (0.55, 0.38), (0.95, 0.9), (0.1, 0.8), (-0.6, 0.5),
              (-0.45, 0), (-0.6, -0.5), (0.1, -0.8), (0.95, -0.9), (0.55, -0.38)]


def _darken(c, f=0.7):
    return tuple(int(v * f) for v in c)


def _mix(a, b, t):
    return tuple(int(x + (y - x) * t) for x, y in zip(a, b))


def _hp_color(ratio):
    if ratio > 0.5:
        return _mix((250, 210, 80), (90, 230, 170), (ratio - 0.5) * 2)
    return _mix((245, 80, 100), (250, 210, 80), ratio * 2)


class Renderer:
    def __init__(self, cfg: GameConfig, caption: str = "Space Rock Shooter"):
        self.cfg = cfg
        if not 0 < cfg.camera_zoom <= 1:
            raise ValueError("camera_zoom은 0보다 크고 1 이하여야 합니다.")
        self.screen = pygame.display.set_mode((cfg.width, cfg.height))
        self.set_zoom(cfg.camera_zoom)
        pygame.display.set_caption(caption)
        self.clock = pygame.time.Clock()
        # 시스템 폰트가 없으면 pygame 기본 폰트로 대체됨 → 실행 파일에서도 안전
        sans = "malgungothic,applegothic,nanumgothic,notosanscjkkr,segoeui,arial"
        self.font_hud = pygame.font.SysFont(sans, 22, bold=True)
        self.font_label = pygame.font.SysFont(sans, 13, bold=True)
        self.font_small = pygame.font.SysFont(sans, 14)
        self.font_big = pygame.font.SysFont(sans, 56, bold=True)
        self.font_card = pygame.font.SysFont(sans, 24, bold=True)
        self.font_body = pygame.font.SysFont(sans, 17)
        self.font_mono = pygame.font.SysFont("consolas,couriernew,monospace", 13)

        self._rand = random.Random(2024)   # 렌더러 전용 (배경 별 배치 등)
        self._glow_cache = {}
        self.images = self._load_images()          # [추가] images 폴더 이미지
        self._bg = self._make_background()
        self.camera_x = self.camera_y = 0
        self._twinkles = [(self._rand.uniform(0, cfg.world_width), self._rand.uniform(0, cfg.world_height),
                           self._rand.uniform(0, 6.28), self._rand.uniform(0.02, 0.07))
                          for _ in range(45 * cfg.world_scale ** 2)]
        self._twinkles = []   # [변경] 배경 이미지에 별이 있으므로 반짝이 별은 끔
        self._shape_sprites = {k: self._make_shape_sprite(k) for k in SHAPE_KINDS}
        self._ship_sprite = self._make_ship_sprite()
        self._terrain_key = None
        self._floor = None        # 배경 + 성운/중력장/소행성 캐시
        self._cloud_layer = None  # 우주선 위에 덮는 가스 구름 레이어 캐시
        self._tick = 0

        # 연출용 기록 (게임 상태와 무관)
        self._last_frame = -1
        self._first_seen = {}   # id -> 처음 본 frame (등장 애니메이션)
        self._prev_hp = {}      # 도형 id -> 이전 hp (피격 깜빡임)
        self._hit_frame = {}    # 도형 id -> 마지막 피격 frame
        self._prev_shapes = {}  # id -> 이전 프레임 도형 (사라짐 감지)
        self._prev_alive = {}   # 적 id -> 이전 alive
        self._ghosts = []       # 파괴 애니메이션: dict(kind, x, y, r, angle, color, age)

    # ------------------------------------------------------------------
    # 빛 번짐 스프라이트 (BLEND_ADD로 더해서 발광 효과)
    # ------------------------------------------------------------------
    def _glow(self, radius, color, intensity=1.0):
        radius = max(2, int(radius))
        key = (radius, color, round(intensity, 2))
        surf = self._glow_cache.get(key)
        if surf is None:
            surf = pygame.Surface((radius * 2, radius * 2))
            surf.fill((0, 0, 0))
            for r in range(radius, 0, -1):
                k = (1 - r / radius) ** 2 * intensity
                pygame.draw.circle(surf, tuple(min(255, int(c * k)) for c in color), (radius, radius), r)
            if len(self._glow_cache) > 600:
                self._glow_cache.clear()
            self._glow_cache[key] = surf
        return surf

    def _blit_glow(self, target, x, y, radius, color, intensity=1.0):
        g = self._glow(radius, color, intensity)
        target.blit(g, (int(x) - g.get_width() // 2, int(y) - g.get_height() // 2),
                    special_flags=pygame.BLEND_ADD)

    # ------------------------------------------------------------------
    # [변경] 배경: images/bg 이미지를 월드 크기에 꽉 차게 (비율 유지, 넘치는 부분은 잘라냄)
    # ------------------------------------------------------------------
    def _make_background(self):
        w, h = self.cfg.world_width, self.cfg.world_height
        src = self.images["bg"]
        sw, sh = src.get_size()
        k = max(w / sw, h / sh)
        scaled = pygame.transform.smoothscale(src, (max(w, math.ceil(sw * k)), max(h, math.ceil(sh * k))))
        surf = pygame.Surface((w, h)).convert()
        surf.blit(scaled, ((w - scaled.get_width()) // 2, (h - scaled.get_height()) // 2))
        pygame.draw.rect(surf, (60, 72, 120), (0, 0, w, h), 2)   # 월드 경계선
        return surf

    # ------------------------------------------------------------------
    # [추가] 이미지 불러오기 + 크기 맞추기
    # ------------------------------------------------------------------
    def _load_images(self):
        images = {}
        for key in IMAGE_FILES:
            path = image_path(key)
            img = pygame.image.load(str(path))
            images[key] = img.convert() if key == "bg" else img.convert_alpha()
        return images

    @staticmethod
    def _fit(img, longest):
        """긴 변이 longest 픽셀이 되도록 비율을 유지해 확대/축소."""
        w, h = img.get_size()
        k = longest / max(w, h)
        return pygame.transform.smoothscale(img, (max(1, round(w * k)), max(1, round(h * k))))

    @staticmethod
    def _flash_copy(img):
        """피격 순간 하얗게 번쩍이는 버전."""
        flash = img.copy()
        flash.fill((110, 110, 110, 0), special_flags=pygame.BLEND_RGBA_ADD)
        return flash

    def _make_ship_sprite(self):
        """spaceship 이미지: 가로 = 반지름 × ship_draw_scale. 이미지의 위쪽이 기수(포구) 방향."""
        img = self.images["spaceship"]
        w, h = img.get_size()
        width = self.cfg.tank_radius * self.cfg.ship_draw_scale
        sprite = pygame.transform.smoothscale(img, (max(1, round(width)), max(1, round(width * h / w))))
        return sprite, self._flash_copy(sprite)

    def _make_guard_tile(self, T):
        """guard 이미지 한 블록: 타일보다 조금 크게 그려 옆 블록과 겹치며 이어지게."""
        size = T + self.cfg.guard_overlap
        block = pygame.transform.smoothscale(self.images["guard"], (size, size))
        shadow = block.copy()
        shadow.fill((0, 0, 0, 255), special_flags=pygame.BLEND_RGBA_MIN)   # 모양 그대로 검게
        shadow.set_alpha(110)
        return block, shadow

    # ------------------------------------------------------------------
    # [변경] 지형: 벽(WALL) → guard 이미지 블록을 이어 붙인 엄폐물
    # ------------------------------------------------------------------
    def _ensure_terrain(self, tm):
        key = (id(tm), tm.version)
        if key == self._terrain_key:
            return
        self._terrain_key = key
        T = tm.tile
        W, H = self.cfg.world_width, self.cfg.world_height
        floor = self._bg.copy()
        block, shadow = self._make_guard_tile(T)
        half = block.get_width() // 2

        # 그림자를 먼저 모두 깔고 → 블록을 위에서 아래 줄 순서로 그려 자연스럽게 겹치게
        walls = [(c, r) for r in range(tm.rows) for c in range(tm.cols) if tm.tiles[r][c] == WALL]
        for c, r in walls:
            cx, cy = int((c + 0.5) * T), int((r + 0.5) * T)
            floor.blit(shadow, (cx - half + 4, cy - half + 6))
        for c, r in walls:
            cx, cy = int((c + 0.5) * T), int((r + 0.5) * T)
            floor.blit(block, (cx - half, cy - half))

        # (기본으로 꺼져 있는) 2단계 지형도 원본처럼 그려 둔다
        add = pygame.Surface((W, H))
        add.fill((0, 0, 0))
        any_mud = False
        for r in range(tm.rows):
            for c in range(tm.cols):
                if tm.tiles[r][c] == MUD:
                    any_mud = True
                    cx, cy = (c + 0.5) * T, (r + 0.5) * T
                    self._blit_glow(add, cx, cy, T * 0.75, COLORS["gravity"], 0.32)
        if any_mud:
            floor.blit(add, (0, 0), special_flags=pygame.BLEND_ADD)
        for (c, r), hp in tm.hp.items():     # 부서지는 벽: guard 블록에 금이 가는 모습
            cx, cy = int((c + 0.5) * T), int((r + 0.5) * T)
            floor.blit(block, (cx - half, cy - half))
            dmg = 1 - hp / tm.breakable_hp_max
            cracks = [((-0.25, -0.2), (0.0, 0.0), (-0.1, 0.3)), ((0.3, -0.15), (0.05, 0.05), (0.25, 0.28)),
                      ((0.0, -0.32), (0.02, -0.05), (-0.28, 0.08))]
            for crack in cracks[:int(dmg * 3.999)]:
                pygame.draw.lines(floor, (20, 24, 36), False, [(cx + x * T, cy + y * T) for x, y in crack], 2)
        self._floor = floor

        # 가스 구름 (기본 꺼짐) 레이어
        layer = pygame.Surface((W, H), pygame.SRCALPHA)
        for r in range(tm.rows):
            for c in range(tm.cols):
                if tm.tiles[r][c] == BUSH:
                    for ox, oy in ((0.28, 0.3), (0.72, 0.28), (0.3, 0.72), (0.7, 0.7), (0.5, 0.5)):
                        self._aa_circle(layer, (c + ox) * T, (r + oy) * T, T * 0.36, COLORS["cloud"])
        layer.set_alpha(240)
        self._cloud_layer = layer

    # ------------------------------------------------------------------
    # [변경] 운석 스프라이트: rock_01 / rock_02 / rock_03 이미지 (한 번 만들어 두고 회전해서 씀)
    # ------------------------------------------------------------------
    def _make_shape_sprite(self, kind):
        r = SHAPE_TYPES[kind]["radius"]
        sprite = self._fit(self.images[kind], r * self.cfg.rock_draw_scale)
        return sprite, self._flash_copy(sprite)

    # ------------------------------------------------------------------
    # [추가] 플레이어 우주선 = spaceship 이미지
    #   게임 각도 angle: 0 = 오른쪽(→), 시계방향 +.  이미지의 위쪽(포구)을 angle 방향으로 돌린다.
    #   pygame.transform.rotate는 반시계방향이 + 이므로  -(angle(도) + 90)  만큼 돌린다.
    # ------------------------------------------------------------------
    def _draw_player_ship(self, t, f):
        c = self.cfg
        x, y = self.world_to_view(t.x, t.y)
        scale = self._spawn_scale(t.id, f)
        if scale < 0.08:
            return
        sprite, flash = self._ship_sprite
        img = flash if self._flash_on and f - t.last_hit_frame < FLASH_FRAMES else sprite

        # 엔진 빛: 기수 반대쪽(이미지 아래쪽)에서, 빠를수록 크게
        ca, sa = math.cos(t.angle), math.sin(t.angle)
        spd = min(1.0, math.hypot(t.vx, t.vy) / (c.max_speed * t.accel_scale))
        back = img.get_height() * 0.42 * scale
        flick = 0.1 * ((self._tick // 2) % 3)
        self._blit_glow(self.screen, x - ca * back, y - sa * back,
                        t.radius * scale * (0.5 + 0.5 * spd + flick), (255, 140, 60), 0.4 + 0.6 * spd)

        img = pygame.transform.rotozoom(img, -(math.degrees(t.angle) + 90), scale)
        self.screen.blit(img, img.get_rect(center=(int(x), int(y))))

        label = self.font_label.render("YOU", True, COLORS["white"])
        pos = label.get_rect(center=(x, y - t.radius - 22))
        self.screen.blit(self.font_label.render("YOU", True, (0, 0, 0)), pos.move(1, 1))
        self.screen.blit(label, pos)

    def _draw_twinkles(self):
        for x, y, phase, speed in self._twinkles:
            b = 0.5 + 0.5 * math.sin(self._tick * speed + phase)
            c = int(90 + 165 * b)
            sx, sy = self.world_to_view(x, y)
            if -2 <= sx <= self.view_width + 2 and -2 <= sy <= self.view_height + 2:
                pygame.draw.circle(self.screen, (c, c, min(255, c + 20)), (int(sx), int(sy)), 1 if b < 0.7 else 2)



    # ------------------------------------------------------------------
    # 기본 도형 (안티앨리어싱)
    # ------------------------------------------------------------------
    def _aa_poly(self, surf, pts, color):
        ipts = [(int(round(x)), int(round(y))) for x, y in pts]
        pygame.gfxdraw.filled_polygon(surf, ipts, color)
        pygame.gfxdraw.aapolygon(surf, ipts, color)

    def _aa_circle(self, surf, x, y, r, color):
        x, y, r = int(round(x)), int(round(y)), max(1, int(round(r)))
        pygame.gfxdraw.filled_circle(surf, x, y, r, color)
        pygame.gfxdraw.aacircle(surf, x, y, r, color)

    def _rect_rot(self, cx, cy, ang, length, half_w, start=0.0):
        ca, sa = math.cos(ang), math.sin(ang)
        x0, y0 = cx + ca * start, cy + sa * start
        x1, y1 = cx + ca * length, cy + sa * length
        return [(x0 - sa * half_w, y0 + ca * half_w), (x0 + sa * half_w, y0 - ca * half_w),
                (x1 + sa * half_w, y1 - ca * half_w), (x1 - sa * half_w, y1 + ca * half_w)]

    # ------------------------------------------------------------------
    # 연출 기록 갱신
    # ------------------------------------------------------------------
    def _track(self, game):
        f = game.frame
        if f < self._last_frame:  # 재시작
            self._first_seen.clear(); self._prev_hp.clear(); self._hit_frame.clear()
            self._prev_shapes.clear(); self._prev_alive.clear(); self._ghosts.clear()
        # 파괴 애니메이션 진행량: 게임 프레임 기준, 단 게임오버로 멈춘 동안에도 끝까지 재생
        delta = f - self._last_frame if f > self._last_frame else (1 if game.game_over else 0)
        for g in self._ghosts:
            g["age"] += delta
        self._last_frame = f

        cur = {s.id: s for s in game.shapes}
        for sid, s in self._prev_shapes.items():
            if sid not in cur:
                self._ghosts.append(dict(kind=s.kind, x=s.x, y=s.y, r=s.radius, angle=s.angle,
                                         color=COLORS[s.kind], age=0))
        for s in game.shapes:
            self._first_seen.setdefault(s.id, f)
            if s.hp < self._prev_hp.get(s.id, s.hp):
                self._hit_frame[s.id] = f
            self._prev_hp[s.id] = s.hp
        self._prev_shapes = {sid: _ShapeSnap(s) for sid, s in cur.items()}

        for e in game.enemies:
            was = self._prev_alive.get(e.id, e.alive)
            if was and not e.alive:
                self._ghosts.append(dict(kind="ship", x=e.x, y=e.y, r=e.radius, angle=e.angle,
                                         color=COLORS["enemy"], age=0))
            if not was and e.alive:
                self._first_seen[e.id] = f
            self._first_seen.setdefault(e.id, f)
            self._prev_alive[e.id] = e.alive
        p = game.player
        if self._prev_alive.get(p.id, True) and not p.alive:
            self._ghosts.append(dict(kind="ship", x=p.x, y=p.y, r=p.radius, angle=p.angle,
                                     color=COLORS["player"], age=0))
        self._prev_alive[p.id] = p.alive

        self._ghosts = [g for g in self._ghosts if g["age"] < DEATH_FRAMES]
        # 오래된 id 정리
        if len(self._first_seen) > 400:
            live = set(cur) | {e.id for e in game.enemies} | {p.id}
            self._first_seen = {k: v for k, v in self._first_seen.items() if k in live}
            self._prev_hp = {k: v for k, v in self._prev_hp.items() if k in live}
            self._hit_frame = {k: v for k, v in self._hit_frame.items() if k in live}

    def _spawn_scale(self, oid, frame):
        t = (frame - self._first_seen.get(oid, -999)) / SPAWN_FRAMES
        if t >= 1:
            return 1.0
        t = max(0.0, t)
        return 1 - (1 - t) ** 3  # ease-out

    # ------------------------------------------------------------------
    # 그리기
    # ------------------------------------------------------------------
    def set_zoom(self, zoom):
        if not 0 < zoom <= 1:
            raise ValueError("camera_zoom은 0보다 크고 1 이하여야 합니다.")
        self.cfg.camera_zoom = zoom
        self.view_width = round(self.cfg.width / zoom)
        self.view_height = round(self.cfg.height / zoom)
        self._view = pygame.Surface((self.view_width, self.view_height)).convert()
        self._fx_layer = pygame.Surface((self.view_width, self.view_height), pygame.SRCALPHA)

    def world_to_view(self, x, y):
        return x - self.camera_x, y - self.camera_y

    def world_to_screen(self, x, y):
        vx, vy = self.world_to_view(x, y)
        return vx * self.cfg.width / self.view_width, vy * self.cfg.height / self.view_height

    def screen_to_world(self, x, y):
        return (x * self.view_width / self.cfg.width + self.camera_x,
                y * self.view_height / self.cfg.height + self.camera_y)

    def _update_camera(self, player):
        c = self.cfg
        self.camera_x = (int(max(0, min(player.x - self.view_width / 2, c.world_width - self.view_width)))
                         if c.world_width >= self.view_width else (c.world_width - self.view_width) // 2)
        self.camera_y = (int(max(0, min(player.y - self.view_height / 2, c.world_height - self.view_height)))
                         if c.world_height >= self.view_height else (c.world_height - self.view_height) // 2)

    def _visible(self, x, y, margin=100):
        sx, sy = self.world_to_view(x, y)
        return -margin <= sx <= self.view_width + margin and -margin <= sy <= self.view_height + margin

    def draw(self, game, debug_lines=None, paused=False, show_ai=False, rewards_open=False,
             show_controls=True, menu=None, selected=-1):
        self._tick += 1
        self._update_camera(game.player)
        self._track(game)
        self._ensure_terrain(game.terrain)
        self._flash_on = not game.game_over  # 게임오버로 멈춘 화면에서는 깜빡임 끔
        display = self.screen
        self.screen = scr = self._view
        scr.fill(COLORS["space_top"])
        scr.blit(self._floor, (-self.camera_x, -self.camera_y))
        self._draw_twinkles()
        f = game.frame
        p = game.player
        player_hidden = p.alive and game.is_hidden(p)

        for s in game.shapes:
            if self._visible(s.x, s.y):
                self._draw_shape(s, f)
        for e in game.enemies:
            if e.alive and self._visible(e.x, e.y):
                self._draw_ship(e, f, ENEMY_HULL, COLORS["enemy"], COLORS["enemy_dark"])
        if p.alive and not player_hidden:
            self._draw_player_ship(p, f)   # [변경] spaceship 이미지
        # 가스 구름은 우주선 위에 덮는다 → 안에 있는 적은 실제로 가려짐
        scr.blit(self._cloud_layer, (-self.camera_x, -self.camera_y))
        if player_hidden:  # 내 우주선만 반투명하게 보여줌
            self._fx_layer.fill((0, 0, 0, 0))
            self.screen = self._fx_layer
            self._draw_player_ship(p, f)
            self.screen = scr
            self._fx_layer.set_alpha(150)
            scr.blit(self._fx_layer, (0, 0))
        for b in game.bullets:
            if self._visible(b.x, b.y):
                self._draw_bullet(b)
        for g in self._ghosts:
            if self._visible(g["x"], g["y"]):
                self._draw_ghost(g)
        # 체력바는 모든 물체 위에 (숨은 적의 체력바는 그리지 않음)
        for s in game.shapes:
            if s.hp < s.max_hp:
                self._hp_bar(s.x, s.y + s.radius + 9, s.radius * 2.2, s.hp / s.max_hp)
        for e in game.enemies:
            if e.alive and e.hp < e.max_hp and not game.is_hidden(e):
                self._hp_bar(e.x, e.y + e.radius + 12, e.radius * 2.6, e.hp / e.max_hp)
        if p.alive and p.hp < p.max_hp:
            self._hp_bar(p.x, p.y + p.radius + 12, p.radius * 2.6, p.hp / p.max_hp)
        if show_ai:
            self._draw_ai_debug(game)

        self.screen = display
        pygame.transform.smoothscale(scr, (self.cfg.width, self.cfg.height), display)
        self._draw_hud(game, paused, show_notice=not rewards_open and menu is None,
                       show_controls=show_controls)
        if debug_lines:
            self._draw_debug(debug_lines)
        if menu is not None:
            self._draw_menu(menu, "일시정지 · 설정", selected)
        elif game.game_over:
            self._draw_game_over(game)
        elif rewards_open:
            self._draw_rewards(game)
        elif paused:
            self._draw_card(["일시정지"], ["P / ESC: 계속"])

    def draw_main_menu(self, buttons, selected=0):
        pygame.transform.smoothscale(self._bg, (self.cfg.width, self.cfg.height), self.screen)
        self._draw_menu(buttons, "SPACE ROCK SHOOTER", selected, main=True)

    def _draw_menu(self, buttons, title, selected, main=False):
        W, H = self.cfg.width, self.cfg.height
        shade = pygame.Surface((W, H), pygame.SRCALPHA)
        shade.fill((0, 0, 15, 170 if main else 215))
        self.screen.blit(shade, (0, 0))
        if not main:
            self._panel(pygame.Rect(W // 2 - 225, H // 2 - 276, 450, 552), 240, 20)
        font = self.font_big if main else self.font_card
        text = font.render(title, True, COLORS["white"])
        self.screen.blit(text, text.get_rect(center=(W // 2, H // 2 - (188 if main else 224))))
        sub = "엄폐물 뒤에 숨어 운석을 부수세요" if main else "설정을 클릭해 변경 · ESC / P로 계속하기"
        text = self.font_body.render(sub, True, COLORS["text"])
        self.screen.blit(text, text.get_rect(center=(W // 2, H // 2 - (132 if main else 184))))
        mouse = pygame.mouse.get_pos()
        for i, button in enumerate(buttons):
            active = button.enabled and (i == selected or button.rect.collidepoint(mouse))
            color = (32, 65, 95) if active else (20, 29, 51)
            edge = COLORS["player"] if active else COLORS["panel_edge"]
            if not button.enabled:
                color, edge = (21, 23, 35), (48, 51, 66)
            pygame.draw.rect(self.screen, color, button.rect, border_radius=10)
            pygame.draw.rect(self.screen, edge, button.rect, 2 if active else 1, border_radius=10)
            text_color = COLORS["white"] if button.enabled else (94, 100, 120)
            label = (self.font_card if main else self.font_body).render(button.label, True, text_color)
            self.screen.blit(label, label.get_rect(center=button.rect.center))
        hint = "클릭 또는 ↑↓ / Enter로 선택" if main else "새 게임은 진행도를 초기화합니다 · 설정은 이번 실행 동안 유지됩니다"
        text = self.font_small.render(hint, True, COLORS["text"])
        self.screen.blit(text, text.get_rect(center=(W // 2, buttons[-1].rect.bottom + 30)))

    def _draw_shape(self, s, f):
        sprite, flash = self._shape_sprites[s.kind]
        img = flash if self._flash_on and f - self._hit_frame.get(s.id, -999) < FLASH_FRAMES else sprite
        scale = self._spawn_scale(s.id, f)
        if scale < 0.08:
            return
        img = pygame.transform.rotozoom(img, -math.degrees(s.angle), scale)
        x, y = self.world_to_view(s.x, s.y)
        self.screen.blit(img, img.get_rect(center=(int(x), int(y))))

    def _draw_bullet(self, b):
        color = COLORS["player_shot"] if b.team == "player" else COLORS["enemy_shot"]
        r = b.radius
        x, y = self.world_to_view(b.x, b.y)
        tail = (x - b.vx * 2.2, y - b.vy * 2.2)
        pygame.draw.line(self.screen, _darken(color, 0.55), tail, (x, y), max(2, int(r * 1.1)))
        self._blit_glow(self.screen, x, y, r * 3.2, color, 0.75)
        self._aa_circle(self.screen, x, y, r * 0.75, _mix(color, (255, 255, 255), 0.6))

    def _draw_ship(self, t, f, hull, color, dark, is_player=False):
        c = self.cfg
        x, y = self.world_to_view(t.x, t.y)
        scale = self._spawn_scale(t.id, f)
        if scale < 0.08:
            return
        if self._flash_on and f - t.last_hit_frame < FLASH_FRAMES:
            color = _mix(color, (255, 255, 255), 0.55)
        R = t.radius * scale
        ca, sa = math.cos(t.angle), math.sin(t.angle)

        def tf(px, py):
            return x + (px * ca - py * sa) * R, y + (px * sa + py * ca) * R

        # 엔진 불꽃: 속도가 빠를수록 길게
        spd = min(1.0, math.hypot(t.vx, t.vy) / (c.max_speed * t.accel_scale))
        flick = 0.08 * ((self._tick // 2) % 3)
        length = 0.35 + 0.55 * spd + flick
        ex, ey = tf(-0.62, 0)
        self._blit_glow(self.screen, ex, ey, R * (0.55 + 0.4 * spd), (255, 140, 60), 0.8)
        self._aa_poly(self.screen, [tf(-0.55, 0.24), tf(-0.55, -0.24), tf(-0.6 - length, 0)], (255, 150, 70))
        self._aa_poly(self.screen, [tf(-0.55, 0.12), tf(-0.55, -0.12), tf(-0.6 - length * 0.6, 0)], (255, 235, 170))

        # 보호막 (충돌 범위 표시)
        pygame.gfxdraw.aacircle(self.screen, int(x), int(y), int(R), (*color, 70))

        # 포 (발사 직후 반동으로 짧아짐)
        recoil = max(0.0, (t.cooldown - (t.fire_cooldown_max - 6)) / 6) * 6
        gun_len = (c.barrel_length - recoil) * scale
        self._aa_poly(self.screen, self._rect_rot(x, y, t.angle, gun_len, 4.5 * scale, R * 0.4), dark)
        self._aa_poly(self.screen, self._rect_rot(x, y, t.angle, gun_len - 1.5, 2.8 * scale, R * 0.4),
                      (200, 210, 228))

        # 선체
        pts = [tf(x, y) for x, y in hull]
        self._aa_poly(self.screen, pts, color)
        pygame.draw.polygon(self.screen, dark, pts, 2)
        panel = [tf(x * 0.55 - 0.05, y * 0.55) for x, y in hull]
        self._aa_poly(self.screen, panel, _mix(color, (255, 255, 255), 0.22))
        # 조종석
        cx, cy = tf(0.28, 0)
        self._aa_circle(self.screen, cx, cy, R * 0.22, dark)
        self._aa_circle(self.screen, cx, cy, R * 0.16, (225, 242, 255))

        if is_player:
            label = self.font_label.render("YOU", True, COLORS["white"])
            pos = label.get_rect(center=(x, y - t.radius - 16))
            self.screen.blit(self.font_label.render("YOU", True, (0, 0, 0)), pos.move(1, 1))
            self.screen.blit(label, pos)

    def _draw_ghost(self, g):
        """파괴 연출: 천체는 커지며 사라지고, 우주선은 폭발."""
        t = g["age"] / DEATH_FRAMES
        x, y = self.world_to_view(g["x"], g["y"])
        if g["kind"] == "ship":
            r = g["r"] * (1 + 2.2 * t)
            self._blit_glow(self.screen, x, y, r * 1.4, (255, 170, 80), round((1 - t) * 8) / 8 * 1.2)
            ring = _mix(g["color"], (255, 255, 255), 0.4)
            pygame.gfxdraw.aacircle(self.screen, int(x), int(y), int(r), (*ring, int(255 * (1 - t))))
            pygame.gfxdraw.aacircle(self.screen, int(x), int(y), int(r) - 1, (*ring, int(200 * (1 - t))))
            return
        sprite = self._shape_sprites[g["kind"]][1]
        img = pygame.transform.rotozoom(sprite, -math.degrees(g["angle"]), 1 + 0.7 * t)
        img.set_alpha(int(255 * (1 - t) ** 1.5))
        self.screen.blit(img, img.get_rect(center=(int(x), int(y))))
        self._blit_glow(self.screen, x, y, g["r"] * 2, g["color"], round((1 - t) * 8) / 8 * 0.6)

    def _hp_bar(self, cx, y, w, ratio, h=6):
        if not self._visible(cx, y):
            return
        cx, y = self.world_to_view(cx, y)
        x = int(cx - w / 2)
        w = int(w)
        pygame.draw.rect(self.screen, COLORS["hp_bg"], (x - 1, y - 1, w + 2, h + 2), border_radius=(h + 2) // 2)
        fill = max(h, int(w * ratio))
        pygame.draw.rect(self.screen, _hp_color(ratio), (x, y, fill, h), border_radius=h // 2)

    def _draw_ai_debug(self, game):
        tm = game.terrain
        p = game.player
        for e in game.enemies:
            if not e.alive:
                continue
            if e.ai_path:
                pts = [self.world_to_view(e.x, e.y)] + [self.world_to_view(*tm.center(c, r)) for c, r in e.ai_path]
                pygame.draw.lines(self.screen, (235, 190, 40), False, pts, 2)
            if e.ai_can_see:
                pygame.draw.line(self.screen, (60, 220, 110), self.world_to_view(e.x, e.y), self.world_to_view(p.x, p.y), 2)
            if e.ai_goal:
                gx, gy = self.world_to_view(*e.ai_goal)
                pygame.draw.circle(self.screen, (235, 190, 40), (int(gx), int(gy)), 5, 2)

    # ------------------------------------------------------------------
    # HUD
    # ------------------------------------------------------------------
    def _panel(self, rect, alpha=170, radius=10):
        surf = pygame.Surface(rect.size, pygame.SRCALPHA)
        pygame.draw.rect(surf, (*COLORS["panel"], alpha), surf.get_rect(), border_radius=radius)
        pygame.draw.rect(surf, (*COLORS["panel_edge"], 150), surf.get_rect(), 1, border_radius=radius)
        self.screen.blit(surf, rect.topleft)

    def _draw_hud(self, game, paused, show_notice=True, show_controls=True):
        p = game.player
        W, H = self.cfg.width, self.cfg.height
        secs = game.frame / self.cfg.fps

        ammo = f"탄약 {p.ammo} / {p.max_ammo}" if game.cfg.limited_ammo else "탄약 무제한"
        rect = pygame.Rect(W - 224, 14, 210, 88)
        self._panel(rect)
        color = (255, 130, 100) if game.cfg.limited_ammo and p.ammo == 0 else COLORS["white"]
        self.screen.blit(self.font_body.render(ammo, True, color), (rect.x + 14, rect.y + 10))
        reloading = game.reload_frames_left > 0
        hint = f"재장전 {game.reload_frames_left / game.cfg.fps:.1f}초" if reloading else "R: 재장전 · ESC: 메뉴"
        self.screen.blit(self.font_small.render(hint, True, COLORS["text"]), (rect.x + 14, rect.y + 42))
        if reloading:
            bar = pygame.Rect(rect.x + 14, rect.y + 70, rect.width - 28, 6)
            pygame.draw.rect(self.screen, COLORS["hp_bg"], bar, border_radius=3)
            bar.width = int(bar.width * (1 - game.reload_frames_left / game.reload_frames_total))
            if bar.width:
                pygame.draw.rect(self.screen, COLORS["player"], bar, border_radius=3)
        if game.cfg.limited_ammo and p.ammo == 0 and not game.game_over and not reloading and show_notice:
            text = self.font_body.render("탄약 소진! [R] 재장전 (2.5초)", True, (255, 165, 100))
            self.screen.blit(text, text.get_rect(midtop=(W / 2, 218)))

        # 좌상단: 점수/처치/시간
        items = [(self.font_label.render(label, True, COLORS["label"]),
                  self.font_hud.render(value, True, COLORS["white"]))
                 for label, value in (("SCORE", f"{game.score:,}"), ("KILLS", str(game.kills)),
                                      ("TIME", f"{int(secs) // 60}:{int(secs) % 60:02d}"))]
        width = 10 + sum(l.get_width() + v.get_width() + 18 for l, v in items)
        rect = pygame.Rect(14, 14, width, 40)
        self._panel(rect)
        x = rect.x + 14
        for lab, val in items:
            self.screen.blit(lab, (x, rect.y + 13))
            self.screen.blit(val, (x + lab.get_width() + 6, rect.y + 6))
            x += lab.get_width() + val.get_width() + 18

        if game.cfg.progression_enabled:
            self._draw_progression(game, show_notice)
        self._draw_rock_info(game, show_notice)   # [추가]

        # 하단 중앙: 선체 내구도
        w, h = 320, 18
        bx, by = W / 2 - w / 2, H - 42
        self._panel(pygame.Rect(bx - 6, by - 6, w + 12, h + 12), 170, 15)
        ratio = p.hp / p.max_hp
        pygame.draw.rect(self.screen, COLORS["hp_bg"], (bx, by, w, h), border_radius=h // 2)
        if ratio > 0:
            col = _hp_color(ratio)
            pygame.draw.rect(self.screen, col, (bx, by, max(h, w * ratio), h), border_radius=h // 2)
            pygame.draw.rect(self.screen, _mix(col, (255, 255, 255), 0.4),
                             (bx + 6, by + 3, max(0, w * ratio - 12), 3), border_radius=2)
        txt = f"HULL  {math.ceil(p.hp)} / {p.max_hp:.0f}"
        r = self.font_label.render(txt, True, COLORS["white"]).get_rect(center=(W / 2, by + h / 2))
        self.screen.blit(self.font_label.render(txt, True, (0, 0, 0)), r.move(1, 1))
        self.screen.blit(self.font_label.render(txt, True, COLORS["white"]), r)

        # 메뉴의 조작 안내 설정을 즉시 반영한다.
        if show_controls:
            hint = ("WASD: 이동   Space/좌클릭: 발사(위쪽)   R: 재장전   L: 보상   ESC: 메뉴" if self.cfg.fixed_up_aim
                    else "WASD: 이동   커서: 조준   Space/좌클릭: 발사   R: 재장전   L: 보상   ESC: 메뉴")
            surf = self.font_small.render(hint, True, COLORS["text"])
            self.screen.blit(surf, surf.get_rect(midbottom=(W / 2, by - 14)))

    def _draw_progression(self, game, show_notice):
        p = game.player
        panel = pygame.Rect(14, 64, 300, 90)
        self._panel(panel)
        title = f"Lv.{game.level}  {JOB_NAMES[p.job]}"
        self.screen.blit(self.font_hud.render(title, True, COLORS["white"]), (28, 72))
        bar = pygame.Rect(28, 107, 270, 10)
        pygame.draw.rect(self.screen, COLORS["hp_bg"], bar, border_radius=5)
        fill = bar.copy()
        fill.width = int(bar.width * game.xp / game.xp_to_next_level)
        if fill.width:
            pygame.draw.rect(self.screen, COLORS["player"], fill, border_radius=5)
        xp = f"경험치 {game.xp} / {game.xp_to_next_level}   보상 {game.pending_rewards}개"
        self.screen.blit(self.font_small.render(xp, True, COLORS["text"]), (28, 126))
        message = ""
        if game.pending_rewards and not game.game_over:
            message = f"레벨 업! [L] 보상 선택 — 남은 보상 {game.pending_rewards}개"
        elif game.frame < game.reward_message_until:
            message = f"{game.last_reward} 적용 완료!"
        if message and show_notice:
            text = self.font_body.render(message, True, (255, 230, 130))
            box = pygame.Rect(0, 0, text.get_width() + 36, 44)
            box.midtop = (self.cfg.width // 2, 168)
            self._panel(box, 220)
            self.screen.blit(text, text.get_rect(center=box.center))

    def _draw_rock_info(self, game, show_notice):
        """[추가] 대형 운석(rock_03)까지 남은 파괴 수 + 운석 점수 안내."""
        x, y = self.cfg.width - 314, 112      # 오른쪽 위, 탄약 패널 아래
        panel = pygame.Rect(x, y, 300, 58)
        self._panel(panel)
        left = game.cfg.big_rock_every - game.big_progress
        line1 = f"부순 운석 {game.rocks_destroyed}개 · 대형 운석까지 {left}개"
        line2 = "점수  소 5 · 중 15 (15발) · 대 30 (45발)"
        self.screen.blit(self.font_small.render(line1, True, COLORS["white"]), (x + 14, y + 9))
        self.screen.blit(self.font_small.render(line2, True, COLORS["text"]), (x + 14, y + 32))
        if show_notice and game.frame < game.big_message_until and not game.game_over:
            text = self.font_card.render("대형 운석 출현!", True, (255, 190, 90))
            self.screen.blit(text, text.get_rect(midtop=(self.cfg.width // 2, 118)))

    def _draw_rewards(self, game):
        W, H = self.cfg.width, self.cfg.height
        shade = pygame.Surface((W, H), pygame.SRCALPHA)
        shade.fill((0, 0, 12, 210))
        self.screen.blit(shade, (0, 0))
        title = self.font_card.render("레벨업 보상 — 카드 한 장을 선택하세요", True, COLORS["white"])
        self.screen.blit(title, title.get_rect(center=(W // 2, H // 2 - 175)))
        gap, margin = 18, 30
        width = (W - 2 * margin - 2 * gap) // 3
        accents = {"강화": (95, 205, 255), "능력치": (178, 150, 255),
                   "회복": (100, 230, 165), "전직": (255, 205, 95)}
        for i, card in enumerate(game.reward_choices):
            rect = pygame.Rect(margin + i * (width + gap), H // 2 - 110, width, 240)
            self._panel(rect, 245, 16)
            color = accents[card.category]
            pygame.draw.rect(self.screen, color, (rect.x + 16, rect.y + 18, width - 32, 4), border_radius=2)
            label = self.font_body.render(f"[{i + 1}]  {card.category}", True, color)
            self.screen.blit(label, (rect.x + 20, rect.y + 38))
            name = self.font_card.render(card.title, True, COLORS["white"])
            self.screen.blit(name, (rect.x + 20, rect.y + 76))
            for j, line in enumerate(card.description):
                text = self.font_small.render(line, True, COLORS["text"])
                self.screen.blit(text, (rect.x + 20, rect.y + 126 + j * 28))
            self.screen.blit(self.font_body.render(f"{i + 1} 키로 선택", True, color), (rect.x + 20, rect.bottom - 44))
        hint = f"전투 일시정지 · 남은 보상 {game.pending_rewards}개 · L / ESC: 닫기 (카드 유지) · 전직은 Lv.3부터"
        text = self.font_small.render(hint, True, COLORS["text"])
        self.screen.blit(text, text.get_rect(center=(W // 2, H // 2 + 166)))

    def _draw_debug(self, lines):
        lh = 16
        w = max(self.font_mono.size(l)[0] for l in lines) + 24
        rect = pygame.Rect(self.cfg.width - w - 14, 14, w, 14 + lh * len(lines))
        self._panel(rect, 200)
        for i, line in enumerate(lines):
            col = (140, 220, 255) if i == 0 else (215, 222, 240)
            self.screen.blit(self.font_mono.render(line, True, col), (rect.x + 12, rect.y + 8 + lh * i))

    def _draw_card(self, titles, subs, stats=None):
        W, H = self.cfg.width, self.cfg.height
        shade = pygame.Surface((W, H), pygame.SRCALPHA)
        shade.fill((0, 0, 10, 130))
        self.screen.blit(shade, (0, 0))
        ch = 140 + (70 if stats else 0)
        card = pygame.Rect(0, 0, 420, ch)
        card.center = (W / 2, H / 2)
        self._panel(card, 225, 18)
        y = card.y + 22
        for t in titles:
            s = self.font_big.render(t, True, COLORS["white"])
            self.screen.blit(s, s.get_rect(midtop=(W / 2, y)))
            y += s.get_height() + 4
        if stats:
            cw = card.width / len(stats)
            for i, (label, value) in enumerate(stats):
                cx = card.x + cw * (i + 0.5)
                v = self.font_hud.render(value, True, COLORS["white"])
                l = self.font_label.render(label, True, COLORS["label"])
                self.screen.blit(v, v.get_rect(midtop=(cx, y + 4)))
                self.screen.blit(l, l.get_rect(midtop=(cx, y + 34)))
            y += 64
        for t in subs:
            s = self.font_small.render(t, True, COLORS["text"])
            self.screen.blit(s, s.get_rect(midtop=(W / 2, y + 4)))
            y += 22

    def _draw_game_over(self, game):
        secs = game.frame / self.cfg.fps
        self._draw_card(["GAME OVER"], ["Enter: 새 게임 · ESC: 메뉴"],
                        stats=[("SCORE", f"{game.score:,}"), ("KILLS", str(game.kills)),
                               ("TIME", f"{int(secs) // 60}:{int(secs) % 60:02d}")])


class _ShapeSnap:
    """사라진 도형의 마지막 모습을 기억하기 위한 복사본."""
    __slots__ = ("kind", "x", "y", "radius", "angle")

    def __init__(self, s):
        self.kind, self.x, self.y, self.radius, self.angle = s.kind, s.x, s.y, s.radius, s.angle

In [8]:
# ==========================================================================
# 6) main.py (복사 + 수정)
#    [변경] 발사: 마우스 왼쪽 버튼 또는 Space
#    [변경] N 키: 엄폐물 배치를 새로 만들어 새 게임
#    [변경] 강화학습 env 의존 제거 → 디버그 표시는 간단한 상태 정보만
#    argparse 대신 run_game(seed=...) 함수로 실행
# ==========================================================================

MAX_FRAMES_PER_TICK = 5  # 느린 PC에서 업데이트가 밀려 멈추는 것 방지


def read_player_input(keys, renderer, mouse_pos, buttons):
    # A/D = 왼쪽/오른쪽, W/S = 위/아래
    target = None if renderer.cfg.fixed_up_aim else renderer.screen_to_world(*mouse_pos)
    dx = keys[pygame.K_d] - keys[pygame.K_a]
    dy = keys[pygame.K_s] - keys[pygame.K_w]
    move = move_from_vector(dx, dy)
    fire = bool(buttons[0]) or bool(keys[pygame.K_SPACE])   # [변경] Space로도 발사
    return encode_action(move, 0), target, fire


def debug_lines(game, obs_cfg, action):
    """[변경] 강화학습 관찰값 대신 간단한 상태 정보."""
    p = game.player
    count = {k: sum(s.kind == k for s in game.shapes) for k in SHAPE_KINDS}
    return [f"action {action}   frame {game.frame}",
            f"player ({p.x:.0f}, {p.y:.0f})  v ({p.vx:+.2f}, {p.vy:+.2f})  hp {p.hp:.0f}",
            f"rocks  " + "  ".join(f"{k}:{v}" for k, v in count.items()),
            f"rock_03 대기 {game.pending_big_rocks}  진행 {game.big_progress}/{game.cfg.big_rock_every}",
            f"enemies seeing you: {sum(e.alive and e.ai_can_see for e in game.enemies)}",
            "(lines: green = enemy sees you, yellow = A* path)"]


class GameApplication:
    """메뉴와 사람 입력을 관리한다. 열린 메뉴에서는 게임 시간을 진행하지 않는다."""
    def __init__(self, game, renderer, seed=None):
        self.game, self.renderer, self.seed = game, renderer, seed
        self.cfg = game.cfg
        self.mode = "main"
        self.running = True
        self.rewards_open = False
        self.show_debug = False
        self.show_controls = True
        self.auto_pause = True
        self.selected = 0
        self.acc = 0.0
        self.action = 0
        self.click_requested = False
        self.suppress_fire = True

    def buttons(self):
        return menu_buttons(self.mode, self.cfg, self.show_debug, self.show_controls, self.auto_pause)

    def activate(self, key):
        self.click_requested = False
        self.suppress_fire = True
        self.acc = 0.0
        if key in ("start", "restart"):
            self.game.reset(self.seed)
            self.mode, self.rewards_open, self.selected = "playing", False, 0
        elif key == "resume":
            self.mode = "playing"
        elif key == "main":
            self.mode, self.rewards_open, self.selected = "main", False, 0
        elif key == "quit":
            self.running = False
        elif key == "zoom":
            zooms = (0.45, 0.6, 0.75, 1.0)
            current = min(range(len(zooms)), key=lambda i: abs(zooms[i] - self.cfg.camera_zoom))
            self.renderer.set_zoom(zooms[(current + 1) % len(zooms)])
        elif key == "controls":
            self.show_controls = not self.show_controls
        elif key == "debug":
            self.show_debug = not self.show_debug
        elif key == "auto_pause":
            self.auto_pause = not self.auto_pause

    def handle_event(self, ev):
        if ev.type == pygame.QUIT:
            self.running = False
            return
        if ev.type == pygame.WINDOWFOCUSLOST and self.auto_pause and self.mode == "playing":
            self.mode, self.selected = "pause", 4
            self.acc = 0.0
            self.click_requested = False
            self.suppress_fire = True
            return
        if ev.type == pygame.KEYDOWN and ev.key in (pygame.K_ESCAPE, pygame.K_p):
            if self.mode == "playing":
                self.mode, self.selected = "pause", 4
                self.acc = 0.0
                self.click_requested = False
                self.suppress_fire = True
            elif self.mode == "pause":
                self.activate("resume")
            return
        if self.mode in ("main", "pause"):
            buttons = self.buttons()
            if ev.type == pygame.MOUSEBUTTONDOWN and ev.button == 1:
                for i, button in enumerate(buttons):
                    if button.enabled and button.rect.collidepoint(ev.pos):
                        self.selected = i
                        self.activate(button.key)
                        break
            elif ev.type == pygame.KEYDOWN:
                enabled = [i for i, button in enumerate(buttons) if button.enabled]
                if ev.key in (pygame.K_UP, pygame.K_DOWN, pygame.K_TAB):
                    direction = -1 if ev.key == pygame.K_UP else 1
                    index = enabled.index(self.selected) if self.selected in enabled else 0
                    self.selected = enabled[(index + direction) % len(enabled)]
                elif ev.key in (pygame.K_RETURN, pygame.K_SPACE):
                    self.activate(buttons[self.selected].key)
                elif ev.key == pygame.K_q and self.mode == "pause":
                    self.activate("quit")
            return
        if ev.type == pygame.KEYDOWN:
            if ev.key == pygame.K_l:
                self.rewards_open = not self.rewards_open and bool(self.game.open_rewards())
                self.acc = 0.0
                self.click_requested = False
                self.suppress_fire = True
                return
            if self.rewards_open:
                choices = (pygame.K_1, pygame.K_2, pygame.K_3)
                if ev.key in choices and self.game.choose_reward(choices.index(ev.key)):
                    self.rewards_open = False
                    self.suppress_fire = True
                return
            if ev.key == pygame.K_r:
                self.game.request_reload()
            elif ev.key == pygame.K_RETURN and self.game.game_over:
                self.activate("restart")
            elif ev.key == pygame.K_F1:
                self.show_debug = not self.show_debug
            elif ev.key == pygame.K_n:            # [변경] 엄폐물 배치 새로 만들기
                self.activate("restart")
        elif ev.type == pygame.MOUSEBUTTONDOWN and ev.button == 1 and not self.rewards_open and not self.suppress_fire:
            self.click_requested = True

    def update(self, elapsed, keys, mouse_pos, buttons):
        if not buttons[0] and not keys[pygame.K_SPACE]:
            self.suppress_fire = False
        if self.mode != "playing" or self.rewards_open or self.game.game_over:
            self.acc = 0.0
            self.click_requested = False
            return
        dt = 1.0 / self.cfg.fps
        self.acc = min(self.acc + elapsed, dt * MAX_FRAMES_PER_TICK)
        while self.acc >= dt:
            self.renderer._update_camera(self.game.player)
            self.action, target, want_fire = read_player_input(keys, self.renderer, mouse_pos, buttons)
            fire = (want_fire or self.click_requested) and not self.suppress_fire
            self.game.step(self.action, want_fire=fire, aim_position=target)
            self.click_requested = False
            self.game.pop_events()
            self.acc -= dt
            if self.game.game_over:
                self.acc = 0.0
                break

    def draw(self, obs_cfg):
        if self.mode == "main":
            self.renderer.draw_main_menu(self.buttons(), self.selected)
            return
        lines = debug_lines(self.game, obs_cfg, self.action) if self.show_debug else None
        self.renderer.draw(self.game, lines, self.mode == "pause", show_ai=self.show_debug,
                           rewards_open=self.rewards_open and self.mode == "playing",
                           show_controls=self.show_controls,
                           menu=self.buttons() if self.mode == "pause" else None, selected=self.selected)


def run_game(seed=None, fixed_up_aim=False):
    """게임 실행. 창을 닫거나 메뉴의 '게임 나가기'를 누르면 이 셀이 끝난다.

    seed         : 숫자를 주면 엄폐물/운석 배치가 매번 같아짐 (None이면 매번 랜덤)
    fixed_up_aim : True면 마우스 조준 없이 항상 위쪽으로만 발사
    """
    pygame.init()
    try:
        cfg = GameConfig(fixed_up_aim=fixed_up_aim)
        obs_cfg = None
        renderer = Renderer(cfg)
        game = GameCore(cfg, seed=seed)
        app = GameApplication(game, renderer, seed)
        while app.running:
            for ev in pygame.event.get():
                app.handle_event(ev)
                if not app.running:
                    break
            if not app.running:
                break
            app.update(renderer.clock.tick(cfg.fps) / 1000.0, pygame.key.get_pressed(),
                       pygame.mouse.get_pos(), pygame.mouse.get_pressed())
            app.draw(obs_cfg)
            pygame.display.flip()
    finally:
        pygame.quit()   # 노트북에서 셀을 다시 실행해도 창이 새로 열리도록 항상 정리

In [9]:
# ==========================================================================
# 7) 게임 실행  ▶ 이 셀을 실행하면 게임 창이 열립니다
#    - 조작: W/A/S/D 이동, 마우스로 조준, 마우스 왼쪽 버튼 또는 Space로 발사
#    - 항상 위쪽으로만 쏘고 싶으면: run_game(fixed_up_aim=True)
#    - 같은 배치로 연습하고 싶으면: run_game(seed=42)
# ==========================================================================
run_game()